# 07. Große Zustandsräume: kompakt speichern, von beiden Seiten suchen, mit Schätzung (A*) und mit Tiefenschranke

In [Notebook 05 von learn_bwinf1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/05_breitensuche.ipynb) und [Notebook 06](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/06_zustaende.ipynb) hast du gelernt: Ein **Zustand** beschreibt alles, was für die Zukunft zählt, jeder **Zug** führt zu einem neuen Zustand, und die **Breitensuche** findet den kürzesten Weg zum Ziel, indem sie Schicht für Schicht vorgeht und schon gesehene Zustände merkt. [Notebook 07 von learn_bwinf1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/07_dijkstra.ipynb) hat dazu die Prioritätswarteschlange (`heapq`) gebracht.

Das reicht, solange alle Zustände in den Speicher passen. In der 2. Runde sind die Zustandsräume oft **viel größer**: Millionen, Milliarden Zustände, oder mehr, als es Atome gibt. Dann entscheidet nicht mehr nur, ob du die Breitensuche kennst, sondern **wie viele Zustände du anfassen musst**.

In diesem Notebook lernst du:

- Zustände **kompakt und eindeutig** zu speichern (Text, Tupel, Zahl) und ihre **Zahl abzuschätzen und zu messen**, bevor du losrechnest, mit einer **Invariante** ([Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/02_struktur_erkennen.ipynb)) als Hilfe, unlösbare Fälle zu erkennen,
- die Suche zu **zählen** und mehrere Verfahren in einer **Tabelle** zu vergleichen (wie in [Notebook 06](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/06_backtracking_gross.ipynb): Verfahren × Beispiel → bearbeitete Zustände),
- **vom Ziel aus zu rechnen**: alle Abstände einmal berechnen und dann jede Anfrage in wenigen Schritten beantworten,
- die **Suche von beiden Seiten** und warum sie so viel spart,
- **A\***: mit einer **zulässigen Schätzung** Zustände bevorzugen, die nach dem Ziel aussehen, und trotzdem **beweisbar** den kürzesten Weg finden. Hier übergibst du der Suche eine **Funktion als Argument**: die Schätzung,
- die **Tiefenschranke** und das **iterative Vertiefen**: wenig Speicher, dafür Wiederholungen. Als Ausblick **IDA\***.

Das neutrale Beispiel ist das **Schiebepuzzle** mit 3 × 3 Feldern (8 Kacheln und eine Lücke). Danach wendest du die Ideen in zwei Ecken an: **Pancake Sort** (41. BwInf) und **Simultane Labyrinthe** (43. BwInf). **Schach dem Wildschwein** (37. BwInf, Gewinnstellungen mit Tiefenschranke und Rückwärtsanalyse) ist nur als Verweis am Schluss aufgeführt, es ist für ein Notebook in der Größe zu lang. Für das Puzzle gibt es am Ende auch einen MiniZinc-Abschnitt.

Die vorigen Notebooks haben dir gezeigt, wie man **Güte misst**. Hier brauchst du das für A\* nur in einer besonderen Form: Das Verfahren selbst ist **exakt** (es findet immer den kürzesten Weg), gemessen wird, **wie viele Zustände** es dafür braucht und **wie gut die Schätzung** ist.

## 1. Zustände kompakt speichern und die Größe des Raums abschätzen

Beim **Schiebepuzzle** liegen 8 Kacheln mit den Nummern 1 bis 8 auf einem 3 × 3-Brett, ein Feld ist leer (die **Lücke**). In einem Zug schiebst du eine Kachel neben der Lücke auf die Lücke, anders gesagt: Die Lücke wandert auf ein Nachbarfeld. Ziel ist

```
1 2 3
4 5 6
7 8 _
```

**Wie speichert man einen Zustand?** Er muss

- **vergleichbar** sein (ist das derselbe Zustand?),
- als **Schlüssel** in einem Wörterbuch oder als Element einer Menge taugen (also **unveränderlich** sein: eine Liste geht nicht, ein Tupel, ein Text oder eine Zahl schon),
- möglichst **wenig Platz** brauchen, wenn es sehr viele sind.

Hier nehmen wir einen **Text aus 9 Ziffern**, Zeile für Zeile, die Lücke ist die `0`: `'123456780'` ist das Ziel. Ein Text mit 9 Zeichen ist kompakter als ein Tupel aus 9 Zahlen, und `zustand[4]` und `zustand.index('0')` gehen trotzdem. Noch kleiner wäre **eine einzige Zahl**: aus `'123456780'` wird `123456780`. Das spart Speicher, aber du musst auf die Nullen am Anfang aufpassen (unten findest du den Fehler). Man kann einen Zustand auch **gleich nummerieren** (in einer Liste statt einem Wörterbuch speichern), wenn man die Zahl der Zustände kennt, wie bei den Labyrinthen in der Ecke.

**Normalform:** Wenn zwei verschiedene Beschreibungen **dasselbe** bedeuten (bei den Pfannkuchen unten: gleiche Reihenfolge, andere Größen), wähle für alle **eine** Darstellung. Sonst zählst du denselben Zustand mehrfach.

In [ ]:
from collections import deque
import heapq

ZIEL = '123456780'          # Der Zustand ist ein Text aus 9 Ziffern: Zeile für Zeile, 0 ist die Lücke.


def zeige(zustand):
    for zeile in range(3):
        teil = zustand[3 * zeile:3 * zeile + 3]
        print(' '.join(teil).replace('0', '_'))


# Zu jedem Feld (0 bis 8) die Nachbarfelder: oben, unten, links, rechts, soweit es sie gibt.
NACHBARFELDER = []
for feld in range(9):
    zeile = feld // 3
    spalte = feld % 3
    liste = []
    if zeile > 0:
        liste.append(feld - 3)
    if zeile < 2:
        liste.append(feld + 3)
    if spalte > 0:
        liste.append(feld - 1)
    if spalte < 2:
        liste.append(feld + 1)
    NACHBARFELDER.append(liste)

zeige('647850321')
print(NACHBARFELDER[0], NACHBARFELDER[4])

#### Aufgabe 1.1

Schreibe `folger(zustand)`: Sie gibt die Liste aller Zustände zurück, die man mit **einem** Zug erreicht. Die Lücke steht auf Feld `luecke = zustand.index('0')`, ihre möglichen Ziele sind `NACHBARFELDER[luecke]`. Ein Text lässt sich nicht ändern; wandle ihn in eine Liste um, tausche zwei Einträge und füge die Liste wieder zu einem Text zusammen (`''.join(liste)`).

**Erwartete Ausgabe:**
```
['123450786', '123456708']
4 2
```

In [ ]:
def folger(zustand):
    """Die Liste aller Zustände, die man mit einem Zug erreicht (die Lücke wandert auf ein Nachbarfeld)."""
    # TODO
    return []


print(folger(ZIEL))
print(len(folger('123406758')), len(folger('012345678')))

<details>
<summary>Tipp anzeigen</summary>

Für jeden Nachbar: `liste = list(zustand)`, dann `liste[luecke], liste[nachbar] = liste[nachbar], liste[luecke]`, dann `''.join(liste)` an die Ergebnisliste hängen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def folger(zustand):
    """Die Liste aller Zustände, die man mit einem Zug erreicht (die Lücke wandert auf ein Nachbarfeld)."""
    luecke = zustand.index('0')
    ergebnis = []
    for nachbar in NACHBARFELDER[luecke]:
        liste = list(zustand)                                   # Text -> Liste, denn Texte kann man nicht ändern
        liste[luecke], liste[nachbar] = liste[nachbar], liste[luecke]
        ergebnis.append(''.join(liste))                         # Liste -> Text
    return ergebnis


print(folger(ZIEL))
print(len(folger('123406758')), len(folger('012345678')))
```

</details>

#### Frage 1.2

Das Schiebepuzzle mit 3 × 3 Feldern hat **9!** mögliche Anordnungen der 9 Zeichen (8 Kacheln und die Lücke).

1. Wie groß ist 9!? (Kopfrechnen oder `math.factorial(9)`.)
2. Das große Brüderchen mit 4 × 4 Feldern (15 Kacheln) hat **16!** Anordnungen. Wie viele sind das ungefähr? Wie viel Speicher bräuchtest du, wenn jeder gesehene Zustand in einem Wörterbuch mindestens **50 Byte** kostet?
3. Warum ist das 4 × 4-Puzzle mit einer normalen Breitensuche nicht zu lösen, das 3 × 3-Puzzle aber schon?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. 9! = 362 880.
2. 16! = 20 922 789 888 000 ≈ 2,1 · 10<sup>13</sup>. Mit 50 Byte je Zustand wären das etwa 10<sup>15</sup> Byte, also **mehr als ein Petabyte**. Dazu kommt, dass nur die Hälfte aller Anordnungen vom Ziel aus erreichbar ist (siehe Aufgabe 1.4), das ändert aber die Größenordnung nicht.
3. Beim 3 × 3-Puzzle passen die 181 440 erreichbaren Zustände (die Hälfte von 9!, siehe unten) locker in den Speicher, die Breitensuche rechnet nur Sekundenbruchteile. Beim 4 × 4-Puzzle ist schon das **Speichern** unmöglich. Dann helfen nur Verfahren, die nur einen kleinen Teil des Raums ansehen (A\*, Tiefenschranke) oder wenig Speicher brauchen (iteratives Vertiefen).

</details>

### Finde den Fehler: Zustand als Zahl

Ein Mitschüler will Speicher sparen und speichert den Zustand `'012368475'` als Zahl. Danach wandelt er sie wieder in einen Text zurück und fragt, ob die Lücke noch da ist. Was geht schief?

In [ ]:
zustand = '012368475'
zahl = int(zustand)               # als Zahl speichern: spart Platz
text = str(zahl)                  # und wieder zurück
print(zahl, text, len(text))
print('0' in text)                # ist die Lücke noch da?

Die Zelle gibt aus:

```
12368475 12368475 8
False
```

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

`int('012368475')` ist die Zahl `12368475`: **führende Nullen gehen verloren**. `str(zahl)` hat nur 8 Zeichen und enthält keine `0` mehr, die Lücke ist verschwunden (und ein späteres `text.index('0')` würde einen Fehler melden). Schlimmer ist der stille Fall: Die Zustände `'012368475'` und `'12368475'` wären als Zahlen **dasselbe**. Abhilfe: beim Zurückwandeln auf 9 Stellen auffüllen (`str(zahl).zfill(9)`) und immer dieselbe Länge benutzen. Oder eine Darstellung wählen, bei der es keine führenden Nullen gibt, zum Beispiel die Kacheln 1 bis 9 mit der Lücke als `9`.

```python
zustand = '012368475'
zahl = int(zustand)
text = str(zahl).zfill(9)         # mit Nullen auf 9 Stellen auffüllen
print(zahl, text, len(text))
print('0' in text)
```

Ausgabe: `12368475 012368475 9` und `True`.

</details>

### Eine Invariante: nicht jeder Zustand ist lösbar

In [Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/02_struktur_erkennen.ipynb) hast du **Invarianten** kennengelernt: Eigenschaften, die ein Zug nie ändert. Das Schiebepuzzle hat eine berühmte. Zähle die **Inversionen**: Paare von Kacheln (die Lücke zählt nicht), bei denen links die **größere** Nummer steht, wenn man die Kacheln Zeile für Zeile hintereinanderschreibt. Bei `'213456780'` ist es genau 1 Paar (die 2 steht vor der 1).

- Ein Zug nach **links oder rechts** ändert die Reihenfolge der Kacheln nicht: Die Inversionen bleiben gleich.
- Ein Zug nach **oben oder unten** springt eine Kachel über **zwei** andere. Jedes dieser beiden Paare wechselt die Reihenfolge, die Zahl der Inversionen ändert sich also um **−2, 0 oder +2**.

Die **Parität** (gerade oder ungerade) der Inversionen bleibt immer gleich. Das Ziel hat 0 Inversionen, also sind **nur Zustände mit gerader Zahl** lösbar: Das ist die **Hälfte** aller Zustände.

#### Aufgabe 1.3

Schreibe `inversionen(zustand)` (zwei Schleifen über alle Paare `i < j`; Paare mit der Lücke `'0'` zählen nicht). `loesbar` ist schon fertig.

**Erwartete Ausgabe:**
```
0 1 0 22
True False True
```

In [ ]:
def inversionen(zustand):
    """Anzahl der Paare von Kacheln (ohne die Lücke), die in falscher Reihenfolge stehen: links die größere Zahl."""
    # TODO
    return 0


def loesbar(zustand):
    return inversionen(zustand) % 2 == 0


print(inversionen(ZIEL), inversionen('213456780'), inversionen('123456078'), inversionen('647850321'))
print(loesbar(ZIEL), loesbar('213456780'), loesbar('647850321'))

<details>
<summary>Tipp anzeigen</summary>

Ein Paar zählt, wenn `zustand[i] != '0'` und `zustand[j] != '0'` und `zustand[i] > zustand[j]`. (Die Kacheln sind Zeichen; Zeichen mit Ziffern vergleicht Python wie die Zahlen.)

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def inversionen(zustand):
    """Anzahl der Paare von Kacheln (ohne die Lücke), die in falscher Reihenfolge stehen: links die größere Zahl."""
    anzahl = 0
    for i in range(9):
        for j in range(i + 1, 9):
            if zustand[i] != '0' and zustand[j] != '0' and zustand[i] > zustand[j]:
                anzahl += 1
    return anzahl


def loesbar(zustand):
    return inversionen(zustand) % 2 == 0


print(inversionen(ZIEL), inversionen('213456780'), inversionen('123456078'), inversionen('647850321'))
print(loesbar(ZIEL), loesbar('213456780'), loesbar('647850321'))
```

</details>

#### Frage 1.4

1. Warum ändert ein Zug nach **links oder rechts** die Zahl der Inversionen nicht, ein Zug nach **oben oder unten** nur um −2, 0 oder +2?
2. Wie hilft dir die Invariante in der Praxis? Was würde eine Breitensuche tun, wenn man ihr einen **unlösbaren** Start gibt?
3. Welche Frage der 2. Runde lässt sich mit einer solchen Invariante **sofort** beantworten, ohne zu suchen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Beim Zug nach links oder rechts tauschen die Lücke und eine Kachel nur ihre Plätze in der **Zeile**. Ohne die Lücke bleibt die Reihenfolge der 8 Kacheln hintereinander dieselbe, also ändert sich keine Inversion. Bei einem Zug nach oben oder unten rutscht eine Kachel in der Reihenfolge über **genau zwei** andere hinweg. Jedes der beiden Paare ändert seine Reihenfolge, entweder von „falsch“ zu „richtig“ (−1) oder umgekehrt (+1). Die Summe aus zwei Änderungen von ±1 ist −2, 0 oder +2.
2. Die Breitensuche durchsucht **alle** erreichbaren Zustände und endet erst, wenn die Schlange leer ist: **181 440** Zustände, bevor sie „keine Lösung“ meldet (siehe unten). Mit der Invariante sagst du **sofort** „unlösbar“ und sparst die ganze Suche. Das ist eine **Schranke aus der Struktur**, wie in Notebook 02.
3. Immer dann, wenn die Aufgabe fragt, ob etwas **überhaupt** möglich ist (zum Beispiel „gibt es eine Lösung?“, „kann man den Zustand erreichen?“) und der Raum riesig ist. Die Invariante beantwortet nur „nein“ sicher; ein „ja“ musst du anders beweisen, etwa mit einer Konstruktion oder einer Suche.

</details>

## 2. Breitensuche mit Zähler, und vom Ziel aus rechnen

Fünf Startzustände, die wir durch das ganze Notebook benutzen: von leicht (A) bis zum **schwersten überhaupt** (E). Dass E am schwersten ist, kennst du erst nach der Rechnung in Aufgabe 2.2.

In [ ]:
# Fünf Startzustände, von leicht bis zum schwersten überhaupt (die Zahl der nötigen Züge steht in der Aufgabe, nicht im Namen).
BEISPIELE = [
    ('A', '162573048'),
    ('B', '263405178'),
    ('C', '387512460'),
    ('D', '518673024'),
    ('E', '647850321'),
]
for name, start in BEISPIELE:
    print(name)
    zeige(start)

Die **Breitensuche** kennst du aus [learn_bwinf1, Notebook 05](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/05_breitensuche.ipynb). Ab jetzt **zählen wir mit**: Das Maß für den Aufwand ist die Zahl der **bearbeiteten Zustände**, also der Zustände, die aus der Warteschlange genommen werden. Das ist nicht von der Geschwindigkeit des Computers abhängig, und im Browser sieht man dieselbe Zahl wie auf dem PC.

#### Aufgabe 2.1

Schreibe `breitensuche(start)` fertig: Sie gibt `(Zahl der Züge, bearbeitete Zustände)` zurück. Das Wörterbuch `abstand` merkt sich zu jedem gesehenen Zustand die Zahl der Züge vom Start.

**Erwartete Ausgabe:**
```
A (10, 599)
B (16, 10655)
C (22, 94146)
D (26, 163696)
E (31, 181440)
```

In [ ]:
def breitensuche(start):
    """Gibt (Zahl der Züge, bearbeitete Zustände) zurück, bei unlösbarem Start (None, bearbeitete Zustände).
    Bearbeitet heißt: aus der Warteschlange genommen."""
    abstand = {start: 0}
    schlange = deque([start])
    bearbeitet = 0
    while schlange:
        zustand = schlange.popleft()
        bearbeitet += 1
        # TODO: Ziel erreicht? Dann (abstand[zustand], bearbeitet) zurückgeben.
        # TODO: alle Folger, die noch nicht in abstand stehen, eintragen und in die Schlange stellen.
    return None, bearbeitet


for name, start in BEISPIELE:
    print(name, breitensuche(start))

<details>
<summary>Tipp anzeigen</summary>

Das ist die Breitensuche aus learn_bwinf1. Wenn `zustand == ZIEL`, gib `abstand[zustand]` und den Zähler zurück. Sonst gehe die `folger(zustand)` durch und stelle jeden, der noch nicht in `abstand` steht, mit `abstand[zustand] + 1` in die Schlange.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def breitensuche(start):
    """Gibt (Zahl der Züge, bearbeitete Zustände) zurück, bei unlösbarem Start (None, bearbeitete Zustände).
    Bearbeitet heißt: aus der Warteschlange genommen."""
    abstand = {start: 0}
    schlange = deque([start])
    bearbeitet = 0
    while schlange:
        zustand = schlange.popleft()
        bearbeitet += 1
        if zustand == ZIEL:
            return abstand[zustand], bearbeitet
        for neu in folger(zustand):
            if neu not in abstand:
                abstand[neu] = abstand[zustand] + 1
                schlange.append(neu)
    return None, bearbeitet


for name, start in BEISPIELE:
    print(name, breitensuche(start))
```

</details>

Die Zahl der Züge steigt von 10 bis 31, die Zahl der bearbeiteten Zustände von 599 auf 181 440. Bei E sind es **alle** erreichbaren Zustände (siehe unten): Die Breitensuche hat praktisch den ganzen Raum durchsucht, um einen einzigen Weg zu finden.

**Und ein unlösbarer Start?** Mit der Invariante aus 1.3 wissen wir, dass `'213456780'` nicht lösbar ist. Die Breitensuche wüsste es nicht, bevor sie alles gesehen hat:

In [ ]:
unloesbar = '213456780'               # ZIEL mit den Kacheln 1 und 2 vertauscht
print(loesbar(unloesbar), breitensuche(unloesbar))

False (None, 181440)


Genau **181 440 = 9! ÷ 2** Zustände wurden bearbeitet, bevor sie „keine Lösung“ melden konnte. Das sind alle Zustände mit gerader Zahl der Inversionen, passend zu unserer Invariante.

### Vom Ziel aus rechnen

Wenn man den ganzen Raum sowieso durchsucht, kann man es **einmal** tun und sich alles merken: Eine Breitensuche **vom Ziel aus**, die nicht aufhört, sondern **alle** erreichbaren Zustände mit ihrem Abstand vom Ziel einträgt. (Die Züge beim Schiebepuzzle sind **umkehrbar**: Wenn ein Zug von X nach Y führt, führt einer von Y nach X. Deshalb sind die Abstände **zum** Ziel dieselben wie die Abstände **vom** Ziel.) Danach braucht jede Anfrage keine Suche mehr: Man geht von jedem Zustand zu einem Folger mit **kleinerem Abstand**, bis man am Ziel ist.

#### Aufgabe 2.2

Schreibe `abstaende_vom_ziel()`: eine Breitensuche, die vom `ZIEL` aus **alle** Zustände besucht und das Wörterbuch `abstand` (Zustand → Zahl der Züge bis zum Ziel) zurückgibt. Die Zellen darunter zählen, wie viele Zustände es sind und wie viele in jeder Entfernung liegen.

**Erwartete Ausgabe:**
```
181440 Zustände erreichbar, weitester Abstand 31 , davon 2 Zustände
Zustände je Abstand ab 0: [1, 2, 4, 8, 16, 20, 39, 62]
```

In [ ]:
def abstaende_vom_ziel():
    """Ein Wörterbuch: zu jedem Zustand, den man vom Ziel aus erreicht, die kleinste Zahl von Zügen bis dorthin."""
    # TODO: Breitensuche vom ZIEL aus, die nicht beim ersten Treffer aufhört, sondern alles besucht.
    return {ZIEL: 0}


abstand_ziel = abstaende_vom_ziel()
je_stufe = {}
for zustand in abstand_ziel:
    d = abstand_ziel[zustand]
    je_stufe[d] = je_stufe.get(d, 0) + 1
weit = max(je_stufe)
print(len(abstand_ziel), 'Zustände erreichbar, weitester Abstand', weit, ', davon', je_stufe[weit], 'Zustände')
print('Zustände je Abstand ab 0:', [je_stufe.get(d, 0) for d in range(8)])

<details>
<summary>Tipp anzeigen</summary>

Wie 2.1, nur ohne Zieltest: Die Schleife läuft, bis die Schlange leer ist.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def abstaende_vom_ziel():
    """Ein Wörterbuch: zu jedem Zustand, den man vom Ziel aus erreicht, die kleinste Zahl von Zügen bis dorthin."""
    abstand = {ZIEL: 0}
    schlange = deque([ZIEL])
    while schlange:
        zustand = schlange.popleft()
        for neu in folger(zustand):
            if neu not in abstand:
                abstand[neu] = abstand[zustand] + 1
                schlange.append(neu)
    return abstand


abstand_ziel = abstaende_vom_ziel()
je_stufe = {}
for zustand in abstand_ziel:
    d = abstand_ziel[zustand]
    je_stufe[d] = je_stufe.get(d, 0) + 1
weit = max(je_stufe)
print(len(abstand_ziel), 'Zustände erreichbar, weitester Abstand', weit, ', davon', je_stufe[weit], 'Zustände')
print('Zustände je Abstand ab 0:', [je_stufe.get(d, 0) for d in range(8)])
```

</details>

Das sind genau die Zahlen, die man beim Schiebepuzzle erwartet: 181 440 erreichbare Zustände, höchstens **31** Züge, und nur **zwei** Zustände brauchen 31 Züge. Einer davon ist unser Beispiel E.

Jetzt löst eine Tabellenabfrage jede Anfrage. Als **Schiedsrichter** ([Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb)) prüfen wir, ob der Weg wirklich zügeweise zum Ziel führt.

#### Aufgabe 2.3

Schreibe `loese_mit_tabelle(start)`: Von `start` aus wählst du immer einen Folger, dessen Abstand in `abstand_ziel` **um 1 kleiner** ist, bis du am Ziel bist. Der Schiedsrichter `pruefe_weg` steht schon da.

**Erwartete Ausgabe:**
```
A 10 Züge, Schiedsrichter: True
B 16 Züge, Schiedsrichter: True
C 22 Züge, Schiedsrichter: True
D 26 Züge, Schiedsrichter: True
E 31 Züge, Schiedsrichter: True
```

In [ ]:
def pruefe_weg(weg):
    """Schiedsrichter: Ist weg eine Liste von Zuständen, die zügeweise von weg[0] zu ZIEL führt?"""
    if weg[-1] != ZIEL:
        return False
    for i in range(len(weg) - 1):
        if weg[i + 1] not in folger(weg[i]):
            return False
    return True


def loese_mit_tabelle(start):
    """Gibt die Liste der Zustände von start bis ZIEL zurück. Benutzt abstand_ziel und keine Suche."""
    weg = [start]
    # TODO: Solange der Zustand nicht ZIEL ist: einen Folger wählen, dessen Abstand um 1 kleiner ist.
    return weg


for name, start in BEISPIELE:
    weg = loese_mit_tabelle(start)
    print(name, len(weg) - 1, 'Züge, Schiedsrichter:', pruefe_weg(weg))

<details>
<summary>Tipp anzeigen</summary>

`while zustand != ZIEL:` und darin eine Schleife über `folger(zustand)`. Der erste Folger mit `abstand_ziel[neu] == abstand_ziel[zustand] - 1` wird der nächste Zustand. Hänge ihn an `weg` und höre mit `break` auf, den Rest der Folger zu betrachten.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def pruefe_weg(weg):
    """Schiedsrichter: Ist weg eine Liste von Zuständen, die zügeweise von weg[0] zu ZIEL führt?"""
    if weg[-1] != ZIEL:
        return False
    for i in range(len(weg) - 1):
        if weg[i + 1] not in folger(weg[i]):
            return False
    return True


def loese_mit_tabelle(start):
    """Gibt die Liste der Zustände von start bis ZIEL zurück. Benutzt abstand_ziel und keine Suche."""
    weg = [start]
    zustand = start
    while zustand != ZIEL:
        for neu in folger(zustand):
            if abstand_ziel[neu] == abstand_ziel[zustand] - 1:
                zustand = neu
                weg.append(neu)
                break
    return weg


for name, start in BEISPIELE:
    weg = loese_mit_tabelle(start)
    print(name, len(weg) - 1, 'Züge, Schiedsrichter:', pruefe_weg(weg))
```

</details>

#### Frage 2.4

1. Wann lohnt es sich, **einmal alle Abstände vom Ziel aus** zu berechnen, und wann nicht?
2. Die Lösung zu 2.3 findet **immer** einen Weg der kürzesten Länge, ohne zu suchen. Warum klappt `while zustand != ZIEL` immer, es gibt also immer einen Folger mit Abstand − 1?
3. Beim 4 × 4-Puzzle (Frage 1.2) geht das nicht. Aber für die **Schätzung** in A\* kann man ein Stück dieser Idee behalten. Wie? (Nur eine Idee, keine Ausführung.)

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Es lohnt sich, wenn es **viele Anfragen** mit demselben Ziel gibt und der Raum **klein genug** ist, um ihn zu speichern. Hier gibt es 181 440 mögliche Starts: Wenn ein Wettbewerbsprogramm für viele Starts die Lösung ausgeben soll, rechnet man die Tabelle einmal. Bei **einem** Start allein ist die Tabelle meist Verschwendung, denn sie besucht auch alle Zustände, die für diesen Start unwichtig sind (bei E war es allerdings fast der ganze Raum).
2. Die Abstände entstehen aus einer Breitensuche vom Ziel: Jeder Zustand mit Abstand d > 0 wurde von einem Zustand mit Abstand d − 1 aus entdeckt. Weil Züge umkehrbar sind, ist dieser Vorgänger auch ein **Folger** des Zustands.
3. Man rechnet die Abstände nicht für das ganze Puzzle, sondern für ein **vereinfachtes** Problem (zum Beispiel nur für die Kacheln 1 bis 4, die anderen sind „egal“). Das Ergebnis ist eine gute **Schätzung**, die man in einer Tabelle nachschlägt (*pattern database*). Das ist die Idee hinter den schnellsten Puzzle-Lösern. In der Ecke zu den Simultanen Labyrinthen machen wir etwas Ähnliches: Die Abstände jeder Person **einzeln** vom Ziel aus sind die Schätzung für beide zusammen.

</details>

## 3. Von beiden Seiten suchen

Eine Breitensuche, die k Züge tief sucht, besucht ungefähr **b<sup>k</sup>** Zustände, wenn jeder Zustand etwa **b** neue Folger hat (die „Verzweigung“). Beim Schiebepuzzle ist b etwas unter 3. Die Zahl wächst also **exponentiell** mit der Tiefe.

Die Idee: Man sucht **gleichzeitig vom Start und vom Ziel aus**, jeweils nur halb so tief. Treffen sich die beiden Suchen in der Mitte, hat man einen Weg. Das kostet etwa **2 · b<sup>k/2</sup>** statt **b<sup>k</sup>** Zustände, bei k = 20 und b = 3 sind das 118 098 statt 3 486 784 401. Anschaulich: Zwei kleine Kreise statt eines großen Kreises, und die Fläche eines Kreises wächst mit dem Quadrat des Radius.

**Voraussetzung:** Man muss vom **Ziel aus rückwärts** gehen können. Beim Schiebepuzzle sind die Züge **umkehrbar**, deshalb ist der Rückwärts-Schritt derselbe wie der Vorwärts-Schritt.

Wir erweitern abwechselnd die Seite, die gerade **weniger Zustände am Rand** hat. Sobald ein neu entdeckter Zustand schon auf der anderen Seite bekannt ist, haben sich die beiden Seiten getroffen: Die Länge ist der Abstand von vorn plus der Abstand von hinten.

#### Aufgabe 3.1

Vervollständige `beidseitig(start)`: Wenn ein neuer Zustand `neu` auch in `andere` steht, gib `(eigene[neu] + andere[neu], bearbeitet)` zurück. Alles andere steht im Gerüst. „Bearbeitet“ zählt hier die Zustände, deren Folger berechnet wurden.

**Erwartete Ausgabe:**
```
A (10, 62)
B (16, 299)
C (22, 1396)
D (26, 3291)
E (31, 10021)
```

In [ ]:
def beidseitig(start):
    """Breitensuche von start und von ZIEL aus, abwechselnd Schicht für Schicht, bis sich beide treffen.
    Gibt (Zahl der Züge, bearbeitete Zustände) zurück. Bearbeitet heißt: Folger berechnet. (Nur für lösbare Starts.)"""
    if start == ZIEL:
        return 0, 0
    vorn = {start: 0}
    hinten = {ZIEL: 0}
    rand_vorn = [start]
    rand_hinten = [ZIEL]
    bearbeitet = 0
    while True:
        # Wir erweitern die Seite, deren Rand kleiner ist.
        if len(rand_vorn) <= len(rand_hinten):
            eigene, andere, rand = vorn, hinten, rand_vorn
        else:
            eigene, andere, rand = hinten, vorn, rand_hinten
        if not rand:
            return None, bearbeitet         # nichts mehr zu erweitern: keine Verbindung (unlösbarer Start)
        neuer_rand = []
        for zustand in rand:
            bearbeitet += 1
            for neu in folger(zustand):
                if neu not in eigene:
                    eigene[neu] = eigene[zustand] + 1
                    neuer_rand.append(neu)
                    # TODO: Steht neu auch in andere? Dann haben sich beide Seiten getroffen:
                    # die Länge ist eigene[neu] + andere[neu].
        rand[:] = neuer_rand            # der alte Rand wird durch den neuen ersetzt (in derselben Liste)


for name, start in BEISPIELE:
    print(name, beidseitig(start))

<details>
<summary>Tipp anzeigen</summary>

Direkt unter `neuer_rand.append(neu)`: `if neu in andere: return eigene[neu] + andere[neu], bearbeitet`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def beidseitig(start):
    """Breitensuche von start und von ZIEL aus, abwechselnd Schicht für Schicht, bis sich beide treffen.
    Gibt (Zahl der Züge, bearbeitete Zustände) zurück. Bearbeitet heißt: Folger berechnet. (Nur für lösbare Starts.)"""
    if start == ZIEL:
        return 0, 0
    vorn = {start: 0}
    hinten = {ZIEL: 0}
    rand_vorn = [start]
    rand_hinten = [ZIEL]
    bearbeitet = 0
    while True:
        # Wir erweitern die Seite, deren Rand kleiner ist.
        if len(rand_vorn) <= len(rand_hinten):
            eigene, andere, rand = vorn, hinten, rand_vorn
        else:
            eigene, andere, rand = hinten, vorn, rand_hinten
        if not rand:
            return None, bearbeitet         # nichts mehr zu erweitern: keine Verbindung (unlösbarer Start)
        neuer_rand = []
        for zustand in rand:
            bearbeitet += 1
            for neu in folger(zustand):
                if neu not in eigene:
                    eigene[neu] = eigene[zustand] + 1
                    neuer_rand.append(neu)
                    if neu in andere:
                        return eigene[neu] + andere[neu], bearbeitet
        rand[:] = neuer_rand            # der alte Rand wird durch den neuen ersetzt (in derselben Liste)


for name, start in BEISPIELE:
    print(name, beidseitig(start))
```

</details>

Dieselben Längen wie bei der Breitensuche: 10, 16, 22, 26, 31. Bearbeitet werden dafür bei E nur noch 10 021 statt 181 440 Zustände. Ein Rest bleibt: Der Raum hat nur 181 440 Zustände, die Breitensuche kann also gar nicht mehr als das bearbeiten. Je größer der Raum, desto größer der Gewinn.

#### Frage 3.2

1. Warum ist die **erste Begegnung** schon ein kürzester Weg? Die Suche hört ja mitten in einer Schicht auf. (Tipp: Was gilt für alle vollständig fertigen Schichten beider Seiten, bevor die aktuelle Schicht angefangen hat?)
2. Warum geht die Suche von beiden Seiten **nicht ohne Weiteres** bei den Pfannkuchen (unten): Dort wird ein Pfannkuchen gegessen. Und bei den Simultanen Labyrinthen: Dort ignoriert eine Person eine Anweisung, wenn eine Wand im Weg ist.
3. Man könnte statt abwechselnd mal vorne, mal hinten auch immer **vorne** eine ganze Schicht und **hinten** eine ganze Schicht erweitern. Welchen Vorteil hat die Regel „die Seite mit dem kleineren Rand“?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Angenommen, vor der aktuellen Schicht hat die vordere Seite alle Zustände bis Tiefe `a` und die hintere alle bis Tiefe `b` vollständig gesammelt, und die beiden Mengen haben **keinen** gemeinsamen Zustand. Dann gibt es keinen Weg mit höchstens `a + b` Zügen (sonst läge ein Zustand auf dem Weg in beiden Mengen). Der kürzeste Weg hat also mindestens `a + b + 1` Züge. Die Begegnung in der aktuellen Schicht liefert einen Weg mit genau `a + 1 + b` Zügen. Er ist also **nicht länger** als jeder andere Weg.
2. Man müsste die Züge **rückwärts** ausführen können. Beim Pfannkuchenessen müsste man einen Pfannkuchen **zurückgeben**, und es ist unklar, welcher Größe (und das Ziel ist nicht ein Zustand, sondern „irgendein sortierter Stapel“). Bei den Labyrinthen führen viele Zustände zum selben Folgezustand, man müsste von einem Zustand zu **allen** Vorgängern rückwärts gehen (bei den Abständen einzelner Felder in der Ecke tun wir das). Die Voraussetzung „Zug umkehrbar, Ziel ein einziger Zustand“ muss man **vor** dem Einsatz prüfen.
3. Wenn eine Seite viel schneller wächst als die andere (zum Beispiel, weil der Raum um das Ziel herum enger ist), bleibt die kleinere Seite lange klein. Es ist billiger, immer die kleinere Seite zu erweitern. Die Gesamtzahl der bearbeiteten Zustände bleibt so kleiner.

</details>

## 4. A*: mit einer Schätzung zum Ziel

Die Breitensuche bearbeitet erst alle Zustände mit 1 Zug, dann alle mit 2 Zügen und so weiter, **egal, in welche Richtung** das Ziel liegt. Bei A\* (gesprochen „A-Stern“) bearbeitest du zuerst den Zustand, der **am vielversprechendsten** aussieht. Dazu braucht man zwei Zahlen:

- **g**: wie viele Züge man bis zu diesem Zustand schon gebraucht hat (genau bekannt),
- **h**: eine **Schätzung**, wie viele Züge es noch bis zum Ziel sind.

Bearbeitet wird immer der Zustand mit dem kleinsten **f = g + h**. Das ist die **Prioritätswarteschlange** aus [learn_bwinf1, Notebook 07](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/07_dijkstra.ipynb) (`heapq`), nur mit der Priorität `g + h` statt `g`. Dijkstra ist der Sonderfall `h = 0` (und bei lauter gleichen Kosten dasselbe wie die Breitensuche).

**Wann bleibt das Ergebnis trotzdem optimal?** Wenn die Schätzung **zulässig** ist: **h(Zustand) ≤ wahre Zahl der Züge bis zum Ziel**, für jeden Zustand. Sie darf zu optimistisch sein (zu klein), aber **nie zu pessimistisch**.

Zwei zulässige Schätzungen für das Schiebepuzzle:

1. **Falsch platzierte Kacheln:** Jede Kachel, die nicht auf ihrem Zielfeld liegt, braucht **mindestens einen** eigenen Zug.
2. **Manhattan-Summe:** Für jede Kachel der Zeilenabstand plus der Spaltenabstand zu ihrem Zielfeld. Jeder Zug bewegt **eine** Kachel um **ein** Feld, also braucht man mindestens so viele Züge, wie die Summe der Abstände.

Die zweite ist **näher an der Wahrheit** (immer mindestens so groß wie die erste) und deshalb besser, solange sie zulässig bleibt.

#### Aufgabe 4.1

Schreibe die beiden Schätzungen `falsch_platziert(zustand)` und `manhattan(zustand)`. Die Lücke zählt in beiden **nicht** mit. Für Kachel `k` auf Feld `feld` ist `ZIEL.index(k)` ihr Zielfeld; Zeile und Spalte eines Feldes sind `feld // 3` und `feld % 3`.

**Erwartete Ausgabe:**
```
A 7 10
B 6 8
C 8 16
D 8 16
E 7 21
0 0 1
```

In [ ]:
def falsch_platziert(zustand):
    """Zahl der Kacheln (ohne die Lücke), die nicht auf ihrem Zielfeld liegen."""
    # TODO
    return 0


def manhattan(zustand):
    """Summe über alle Kacheln (ohne die Lücke): Zeilenabstand plus Spaltenabstand zum Zielfeld."""
    # TODO
    return 0


for name, start in BEISPIELE:
    print(name, falsch_platziert(start), manhattan(start))
print(falsch_platziert(ZIEL), manhattan(ZIEL), manhattan('123456708'))

<details>
<summary>Tipp anzeigen</summary>

Manhattan: Schleife über die 9 Felder, für jede Kachel (nicht `'0'`) `ziel_feld = ZIEL.index(kachel)` und `abs(feld // 3 - ziel_feld // 3) + abs(feld % 3 - ziel_feld % 3)` addieren.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def falsch_platziert(zustand):
    """Zahl der Kacheln (ohne die Lücke), die nicht auf ihrem Zielfeld liegen."""
    anzahl = 0
    for feld in range(9):
        if zustand[feld] != '0' and zustand[feld] != ZIEL[feld]:
            anzahl += 1
    return anzahl


def manhattan(zustand):
    """Summe über alle Kacheln (ohne die Lücke): Zeilenabstand plus Spaltenabstand zum Zielfeld."""
    summe = 0
    for feld in range(9):
        kachel = zustand[feld]
        if kachel != '0':
            ziel_feld = ZIEL.index(kachel)
            summe += abs(feld // 3 - ziel_feld // 3) + abs(feld % 3 - ziel_feld % 3)
    return summe


for name, start in BEISPIELE:
    print(name, falsch_platziert(start), manhattan(start))
print(falsch_platziert(ZIEL), manhattan(ZIEL), manhattan('123456708'))
```

</details>

Ob die beiden Schätzungen wirklich zulässig und wie gut sie sind, lässt sich mit der Tabelle aus Aufgabe 2.2 **messen**: Wir haben für **jeden** Zustand den wahren Abstand. Das ist die **Güte** der Schätzung. (Für ein Rätsel mit riesigem Raum, wo man die wahren Werte nicht kennt, kann man das nur an kleinen Fällen prüfen, wie bei Pancake Sort unten.)

In [ ]:
# Wie gut sind die Schätzungen? Wir vergleichen sie mit den wahren Abständen aus abstand_ziel (jeder 5. Zustand).
alle = list(abstand_ziel)
summe_d = 0
summe_falsch = 0
summe_manh = 0
anzahl = 0
zulaessig = True
for i in range(0, len(alle), 5):
    zustand = alle[i]
    d = abstand_ziel[zustand]
    h1 = falsch_platziert(zustand)
    h2 = manhattan(zustand)
    if h1 > d or h2 > d:
        zulaessig = False
    summe_d += d
    summe_falsch += h1
    summe_manh += h2
    anzahl += 1
print('Mittelwert des wahren Abstands:      %.2f' % (summe_d / anzahl))
print('Mittelwert falsch platziert:         %.2f' % (summe_falsch / anzahl))
print('Mittelwert Manhattan-Summe:          %.2f' % (summe_manh / anzahl))
print('Nie größer als der wahre Abstand:', zulaessig)

Mittelwert des wahren Abstands:      21.97
Mittelwert falsch platziert:         7.10
Mittelwert Manhattan-Summe:          14.00
Nie größer als der wahre Abstand: True


Im Mittel ist der wahre Abstand **21,97** Züge, die Schätzung „falsch platziert“ sagt im Mittel **7,10**, die Manhattan-Summe **14,00**. Keine der beiden ist je größer als der wahre Abstand, jede ist also (zumindest für die geprüften Zustände) **zulässig**. Manhattan ist deutlich näher an der Wahrheit.

#### Aufgabe 4.2

Schreibe `astern(start, schaetzung)` fertig. Die Funktion `schaetzung` wird als **Argument** übergeben, also zum Beispiel `astern(start, manhattan)` (ohne Klammern: du gibst die Funktion selbst weiter, nicht ihr Ergebnis). In der Warteschlange stehen Tupel `(f, g, zustand)`: Python vergleicht Tupel von links nach rechts, also entscheidet zuerst `f`, bei Gleichheit `g`, dann der Zustand.

**Erwartete Ausgabe:**
```
A (10, 16)
B (16, 260)
C (22, 482)
```

In [ ]:
def astern(start, schaetzung):
    """A*-Suche. schaetzung ist eine Funktion: schaetzung(zustand) ist die geschätzte Zahl der Züge bis ZIEL.
    Gibt (Zahl der Züge, bearbeitete Zustände) zurück."""
    beste = {start: 0}                                        # kleinstes bisher bekanntes g je Zustand
    warteschlange = [(schaetzung(start), 0, start)]           # Einträge (f, g, zustand) mit f = g + h
    bearbeitet = 0
    while warteschlange:
        f, g, zustand = heapq.heappop(warteschlange)
        if g > beste[zustand]:
            continue                                          # veralteter Eintrag: es gibt schon einen besseren Weg
        bearbeitet += 1
        # TODO: Ziel? Dann (g, bearbeitet) zurückgeben.
        # TODO: Für jeden Folger den Weg g + 1 prüfen: ist er besser als beste[neu] (oder ist neu unbekannt),
        #       dann eintragen und mit dem Schlüssel (g + 1 + schaetzung(neu), g + 1, neu) in die Warteschlange.
    return None, bearbeitet


for name, start in BEISPIELE[:3]:
    print(name, astern(start, manhattan))

<details>
<summary>Tipp anzeigen</summary>

Wie Dijkstra: Beim Herausnehmen den Zieltest machen (nicht beim Einfügen!). Für jeden Folger `neu`: wenn `g + 1` besser als `beste.get(neu)` ist, eintragen und mit `(g + 1 + schaetzung(neu), g + 1, neu)` einfügen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def astern(start, schaetzung):
    """A*-Suche. schaetzung ist eine Funktion: schaetzung(zustand) ist die geschätzte Zahl der Züge bis ZIEL.
    Gibt (Zahl der Züge, bearbeitete Zustände) zurück."""
    beste = {start: 0}                                        # kleinstes bisher bekanntes g je Zustand
    warteschlange = [(schaetzung(start), 0, start)]           # Einträge (f, g, zustand) mit f = g + h
    bearbeitet = 0
    while warteschlange:
        f, g, zustand = heapq.heappop(warteschlange)
        if g > beste[zustand]:
            continue                                          # veralteter Eintrag: es gibt schon einen besseren Weg
        bearbeitet += 1
        if zustand == ZIEL:
            return g, bearbeitet
        for neu in folger(zustand):
            if neu not in beste or g + 1 < beste[neu]:
                beste[neu] = g + 1
                heapq.heappush(warteschlange, (g + 1 + schaetzung(neu), g + 1, neu))
    return None, bearbeitet


for name, start in BEISPIELE[:3]:
    print(name, astern(start, manhattan))
```

</details>

### Die Tabelle: Verfahren × Beispiel → bearbeitete Zustände

Alle vier Verfahren in einer Tabelle: Breitensuche, beidseitige Breitensuche und A\* mit den beiden Schätzungen. Die Funktion prüft nebenbei, dass alle dieselbe Länge finden. (Die Zahlen sind nicht ganz vergleichbar: „bearbeitet“ heißt bei der beidseitigen Suche, dass Folger berechnet wurden; bei den anderen, dass ein Zustand aus der Warteschlange kam. Die Größenordnung stimmt.)

In [ ]:
def tabelle_puzzle():
    """Bearbeitete Zustände je Verfahren und Beispiel (alle Verfahren liefern dieselbe Länge, das prüft die Funktion)."""
    verfahren = ['Breitensuche', 'beidseitig', 'A* falsch plaz.', 'A* Manhattan']
    print('Bsp. Züge ' + ''.join('%17s' % v for v in verfahren))
    zeilen = []
    for name, start in BEISPIELE:
        laenge, n1 = breitensuche(start)
        l2, n2 = beidseitig(start)
        l3, n3 = astern(start, falsch_platziert)
        l4, n4 = astern(start, manhattan)
        assert laenge == l2 == l3 == l4
        zeilen.append((name, laenge, [n1, n2, n3, n4]))
        print('%-4s %4d ' % (name, laenge) + ''.join('%17s' % ('{:,}'.format(n).replace(',', ' ')) for n in [n1, n2, n3, n4]))
    return zeilen


tab_puzzle = tabelle_puzzle()

Bsp. Züge      Breitensuche       beidseitig  A* falsch plaz.     A* Manhattan
A      10               599               62               33               16
B      16            10 655              299              763              260
C      22            94 146            1 396            7 263              482
D      26           163 696            3 291           38 311            4 054
E      31           181 440           10 021          143 849           21 198


**Lies die Tabelle:**

- Bei **A** (10 Züge) bearbeitet A\* mit der Manhattan-Summe nur **16** Zustände, die Breitensuche **599**. Bei **D** (26 Züge) sind es 4 054 gegen 163 696.
- Die schwache Schätzung (falsch platziert) spart weniger: bei D 38 311 Zustände. Wer näher an der Wahrheit schätzt, muss **weniger** Zustände bearbeiten.
- Die **beidseitige** Suche braucht bei D 3 291 Zustände. Bei den schweren Beispielen D und E ist sie hier sogar **besser** als A\* mit Manhattan (10 021 gegen 21 198 bei E). A\* ist also **nicht immer** das beste Verfahren: Bei diesen schweren Beispielen ist die Schätzung noch weit von der Wahrheit entfernt, und viele Zustände mit kleinem g + h müssen bearbeitet werden. In der Praxis kombiniert man die Verfahren oder nimmt eine **bessere Schätzung** (die Tabelle aus der Frage 2.4 Teil 3).
- Bei E ist die Schätzung 21, der wahre Abstand 31: Es bleibt eine **Lücke von 10 Zügen**, die A\* durch Suchen schließen muss.

#### Frage 4.3

1. Warum ist das Ergebnis von A\* **optimal**, wenn die Schätzung zulässig ist? Betrachte den Moment, in dem A\* das Ziel aus der Warteschlange nimmt. Was gilt für alle anderen Einträge in der Warteschlange? (Tipp: Auf dem wahren kürzesten Weg liegt immer ein Zustand in der Warteschlange, dessen f höchstens so groß ist wie die wahre Länge L.)
2. Warum prüft man das Ziel erst beim **Herausnehmen** aus der Warteschlange und nicht schon beim Einfügen?
3. Zwei zulässige Schätzungen h1 ≤ h2 (überall). Warum bearbeitet A\* mit h2 **höchstens** so viele Zustände wie mit h1 (die Reihenfolge bei Gleichstand ausgenommen)?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Angenommen, A\* nimmt das Ziel mit g = G heraus (f = G, denn h(Ziel) = 0), aber es gibt einen kürzeren Weg der Länge L < G. Auf diesem Weg liegt dann ein Zustand X in der Warteschlange, zu dem A\* schon den richtigen Weg kennt (der erste Zustand des Weges, der noch nicht bearbeitet wurde). Für ihn gilt f(X) = g(X) + h(X) ≤ g(X) + (wahre Restlänge) = L < G, weil h zulässig ist. X hätte also ein **kleineres f** als das Ziel und wäre **vorher** herausgenommen worden. Das ist ein Widerspruch: Es gibt keinen kürzeren Weg.
2. Beim Einfügen kennt man nur **einen** Weg zum Ziel, vielleicht nicht den kürzesten. Erst wenn das Ziel das **kleinste f** hat, kann kein anderer Weg mehr kürzer sein. Wer das Ziel schon beim Einfügen zurückgibt, bekommt eventuell einen zu langen Weg.
3. A\* bearbeitet jeden Zustand mit f < L (L = wahre Länge) in jedem Fall. Mit der größeren Schätzung h2 ist f = g + h2 für jeden Zustand **mindestens** so groß wie mit h1, also gibt es weniger Zustände mit f < L. Eine größere zulässige Schätzung ist **immer** mindestens so gut. Aber sie darf **nie** über den wahren Wert hinausgehen: Dann ist sie nicht mehr zulässig (siehe den nächsten Fehler).

</details>

### Finde den Fehler: Eine zu große Schätzung

Ein Mitschüler denkt: „Wenn eine bessere Schätzung weniger Zustände bearbeitet, nehme ich einfach das **Doppelte** der Manhattan-Summe!“ Er probiert es bei C und D. Stimmt das Ergebnis?

In [ ]:
def doppelt_manhattan(zustand):
    return 2 * manhattan(zustand)          # "noch besser": doppelt so schlau geschätzt?


for name, start in BEISPIELE[2:4]:
    laenge, bearbeitet = astern(start, doppelt_manhattan)
    print(name, 'A* mit 2 * Manhattan:', laenge, 'Züge,', bearbeitet, 'bearbeitet;',
          ' wahre Länge:', abstand_ziel[start], 'Züge')

C A* mit 2 * Manhattan: 30 Züge, 508 bearbeitet;  wahre Länge: 22 Züge
D A* mit 2 * Manhattan: 30 Züge, 748 bearbeitet;  wahre Länge: 26 Züge


<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Die Weglängen 30 statt 22 (bei C) und 30 statt 26 (bei D) sind **zu lang**: Das Verfahren ist nicht mehr optimal. Die Schätzung `2 * manhattan` ist **nicht zulässig**: Sie kann größer sein als der wahre Abstand. Dann kann A\* einen Zustand, der auf dem kürzesten Weg liegt, mit einem zu großen f in der Warteschlange weit hinten liegen lassen und das Ziel auf einem längeren Weg früher herausnehmen. Der Fehler fällt nur auf, wenn man das Ergebnis mit etwas Exaktem vergleicht, hier mit `abstand_ziel`. **Merke:** Eine Schätzung, die A\* beschleunigt, aber nicht zulässig ist, macht aus einem exakten Verfahren eine **Heuristik**. Das darf man tun (es kann viel schneller sein), aber dann muss man **die Güte messen** (Notebook 03) und dem Leser sagen, dass die Länge nicht garantiert optimal ist.

</details>

## 5. Tiefenschranke und iteratives Vertiefen: wenig Speicher

Die Breitensuche und A\* merken sich **alle gesehenen Zustände**. Bei einem riesigen Raum ist das der Engpass, nicht die Zeit. Eine **Tiefensuche** (Backtracking, [Notebook 06](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/06_backtracking_gross.ipynb)) merkt sich nur den **aktuellen Weg**, braucht also fast keinen Speicher. Aber sie läuft in einem Raum mit Zyklen unter Umständen ewig in die Tiefe. Die **Tiefenschranke** verhindert das: Man erlaubt höchstens `rest` Züge und gibt auf, wenn sie aufgebraucht sind.

Welche Schranke nimmt man? **Iteratives Vertiefen:** Probiere die Schranke 0, dann 1, dann 2, … Die erste Schranke, mit der es klappt, ist die **kleinste Zugzahl**: wie die Breitensuche, aber mit dem Speicherbedarf einer Tiefensuche. Der Preis: Die kleinen Schichten werden **mehrfach** durchsucht, und Zustände, die man auf verschiedenen Wegen erreicht, werden mehrfach bearbeitet (wir merken sie uns ja nicht). Wir lassen nur den Zug **gleich wieder zurück** weg.

#### Aufgabe 5.1

Schreibe `tiefensuche(zustand, rest, vorher, zaehler)` fertig: Probiere alle Folger außer `vorher` mit `rest - 1`. Ein Treffer genügt.

**Erwartete Ausgabe:**
```
True 4
False 1
True 6
```

In [ ]:
def tiefensuche(zustand, rest, vorher, zaehler):
    """Gibt True zurück, wenn man von zustand aus ZIEL mit höchstens rest Zügen erreicht.
    vorher ist der Zustand, aus dem wir gerade kamen (wird nicht gleich wieder besucht).
    zaehler ist eine Liste mit einer Zahl: zaehler[0] zählt die Aufrufe."""
    zaehler[0] += 1
    if zustand == ZIEL:
        return True
    if rest == 0:
        return False
    # TODO: alle Folger außer vorher ausprobieren, mit rest - 1. Ein Treffer genügt.
    return False


zaehler = [0]
print(tiefensuche('123456708', 1, None, zaehler), zaehler[0])        # ein Zug vom Ziel entfernt
zaehler = [0]
print(tiefensuche('123456708', 0, None, zaehler), zaehler[0])
zaehler = [0]
print(tiefensuche('123450786', 3, None, zaehler), zaehler[0])

<details>
<summary>Tipp anzeigen</summary>

`for neu in folger(zustand): if neu != vorher and tiefensuche(neu, rest - 1, zustand, zaehler): return True`. Danach `return False`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def tiefensuche(zustand, rest, vorher, zaehler):
    """Gibt True zurück, wenn man von zustand aus ZIEL mit höchstens rest Zügen erreicht.
    vorher ist der Zustand, aus dem wir gerade kamen (wird nicht gleich wieder besucht).
    zaehler ist eine Liste mit einer Zahl: zaehler[0] zählt die Aufrufe."""
    zaehler[0] += 1
    if zustand == ZIEL:
        return True
    if rest == 0:
        return False
    for neu in folger(zustand):
        if neu != vorher and tiefensuche(neu, rest - 1, zustand, zaehler):
            return True
    return False


zaehler = [0]
print(tiefensuche('123456708', 1, None, zaehler), zaehler[0])        # ein Zug vom Ziel entfernt
zaehler = [0]
print(tiefensuche('123456708', 0, None, zaehler), zaehler[0])
zaehler = [0]
print(tiefensuche('123450786', 3, None, zaehler), zaehler[0])
```

</details>

#### Aufgabe 5.2

Schreibe `iterativ_vertiefen(start)` fertig: Für `schranke = 0, 1, 2, …` rufe `tiefensuche(start, schranke, None, zaehler)` auf (mit einem frischen `zaehler = [0]`). Die Funktion gibt `(schranke, alle Aufrufe, Aufrufe der letzten Runde)` zurück.

**Erwartete Ausgabe:**
```
A (10, 1661, 525)
B (16, 49726, 7722)
```

In [ ]:
def iterativ_vertiefen(start):
    """Probiert die Schranke 0, 1, 2, ... bis es klappt. Gibt (Zahl der Züge, alle Aufrufe, Aufrufe der letzten Runde) zurück."""
    insgesamt = 0
    schranke = 0
    while schranke < 40:
        zaehler = [0]
        # TODO: tiefensuche mit dieser Schranke starten. Klappt es, die Ergebnisse zurückgeben.
        insgesamt += zaehler[0]
        schranke += 1
    return None, insgesamt, 0


for name, start in BEISPIELE[:2]:
    print(name, iterativ_vertiefen(start))

<details>
<summary>Tipp anzeigen</summary>

Die Zeile mit `# TODO` ersetzt du durch `gefunden = tiefensuche(start, schranke, None, zaehler)`. Zähle `insgesamt += zaehler[0]` und gib bei `gefunden` `schranke, insgesamt, zaehler[0]` zurück.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def iterativ_vertiefen(start):
    """Probiert die Schranke 0, 1, 2, ... bis es klappt. Gibt (Zahl der Züge, alle Aufrufe, Aufrufe der letzten Runde) zurück."""
    insgesamt = 0
    schranke = 0
    while schranke < 40:
        zaehler = [0]
        gefunden = tiefensuche(start, schranke, None, zaehler)
        insgesamt += zaehler[0]
        if gefunden:
            return schranke, insgesamt, zaehler[0]
        schranke += 1
    return None, insgesamt, 0


for name, start in BEISPIELE[:2]:
    print(name, iterativ_vertiefen(start))
```

</details>

Bei A (10 Züge) wird `tiefensuche` insgesamt 1 661-mal aufgerufen, bei B (16 Züge) 49 726-mal, die Breitensuche hat dort 599 und 10 655 Zustände bearbeitet. Es sind **mehr** Aufrufe, weil Zustände mehrfach vorkommen. Dafür liegen nie mehr als 16 Zustände gleichzeitig im Speicher. Bei C (22 Züge) sind es schon über eine Million Aufrufe (Rechenzeit!), bei D und E ohne Schätzung zu viele für ein Notebook.

#### Frage 5.3

1. In der Ausgabe oben: Wie viel Prozent der Aufrufe fallen bei B in die **letzte** Runde, also die mit der richtigen Schranke? Um welchen Faktor ist das iterative Vertiefen dadurch teurer als ein Durchgang mit der richtigen Schranke allein? Wann fallen die Wiederholungen kaum noch ins Gewicht?
2. Welches Verfahren würdest du für ein Rätsel mit **1 000 000 000 000** Zuständen und einem kurzen Weg (etwa 20 Züge) wählen, wenn dein Speicher nur 1 GB hat? Warum?
3. Wo liegt der Nachteil gegenüber der Breitensuche mit Wörterbuch, wenn man **viele verschiedene Wege** zu denselben Zuständen hat?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei B sind es 7 722 von 49 726 Aufrufen, also etwa **16 %**. Das iterative Vertiefen ist damit etwa **6,4-mal** so teuer wie ein einziger Durchgang mit der richtigen Schranke. Die Zahl der Aufrufe wächst mit jeder Schranke auf etwa das 1,6-Fache (aus den Zahlen für A und B geschätzt). Wächst sie schneller (größere Verzweigung, z. B. Faktor 3 oder mehr), ist die letzte Runde größer als alle früheren zusammen, und die Wiederholungen kosten nur noch einen kleinen Bruchteil.
2. Iteratives Vertiefen (oder IDA\*, siehe unten): Es speichert nur den aktuellen Weg (20 Zustände). Die Breitensuche und A\* müssten sich sehr viele Zustände merken. Wenn die Tiefe klein bleibt, dauert die Suche zwar länger, aber sie **läuft** überhaupt.
3. Ohne Merken wird derselbe Zustand **jedes Mal neu** bearbeitet, wenn man ihn auf einem anderen Weg erreicht. In einem Raum mit vielen kurzen Zyklen (viele Wege zu denselben Zuständen) kann das **sehr viel** mehr Arbeit sein als die Breitensuche mit Wörterbuch. Man tauscht **Speicher gegen Zeit**.

</details>

### Für Schnelle: IDA\*

Das iterative Vertiefen lässt sich mit der Schätzung aus Abschnitt 4 verbinden. Statt „höchstens `rest` Züge“ gilt die Schranke für **f = g + h**: Ein Zweig wird abgeschnitten, sobald `g + h` über der Schranke liegt. Die nächste Schranke ist **das kleinste f, das in dieser Runde über der Schranke lag**. Das ist **IDA\***: Es findet wie A\* bei zulässiger Schätzung den kürzesten Weg, braucht aber nur den Speicher des aktuellen Weges. Weil die Schätzung viele Zweige früh abschneidet, ist die Zahl der Aufrufe oft **kleiner** als beim einfachen iterativen Vertiefen, bei guter Schätzung oft um Größenordnungen kleiner. Sie liegt aber meist über der Zahl der bearbeiteten Zustände von A\*, weil Zustände mehrfach vorkommen.

In [ ]:
def ida_stern(start, schaetzung):
    """IDA*: iteratives Vertiefen mit der Schranke f = g + h. Gibt (Zahl der Züge, Aufrufe) zurück."""
    zaehler = [0]

    def suche(zustand, g, vorher, schranke):
        """Gibt -1 zurück, wenn ZIEL gefunden wurde, sonst das kleinste f, das über der Schranke lag."""
        zaehler[0] += 1
        f = g + schaetzung(zustand)
        if f > schranke:
            return f
        if zustand == ZIEL:
            return -1
        kleinstes = 1000
        for neu in folger(zustand):
            if neu != vorher:
                ergebnis = suche(neu, g + 1, zustand, schranke)
                if ergebnis == -1:
                    return -1
                kleinstes = min(kleinstes, ergebnis)
        return kleinstes

    schranke = schaetzung(start)
    while True:
        ergebnis = suche(start, 0, None, schranke)
        if ergebnis == -1:
            return schranke, zaehler[0]
        schranke = ergebnis                 # nächste Runde: die kleinste Schranke, die etwas Neues zulässt


print('Beispiel  Züge  Aufrufe')
ida_ergebnisse = []
for name, start in BEISPIELE[:4]:
    laenge, aufrufe = ida_stern(start, manhattan)
    ida_ergebnisse.append((name, laenge, aufrufe))
    print('%-8s %5d %8d' % (name, laenge, aufrufe))

Beispiel  Züge  Aufrufe
A           10       18
B           16      488
C           22      990
D           26     8723


Mit der Manhattan-Summe braucht IDA\* bei B nur 488 Aufrufe (statt 49 726 beim einfachen Vertiefen) und bei D (26 Züge) 8 723, bei einem Speicher von höchstens 26 Zuständen. Auch hier gilt: Die Schätzung muss **zulässig** sein, sonst ist die Länge nicht garantiert optimal.

## Wettbewerbs-Ecke 1: Pancake Sort

*Angelehnt an Aufgabe 3 „Pancake Sort“ der 2. Runde des 41. Bundeswettbewerbs Informatik (2022/23).*

Ein Stapel Pfannkuchen unterschiedlicher Größe soll sortiert werden, der kleinste soll oben liegen. Du hast einen Wender. Eine **Wende-und-Ess-Operation**: Der Wender wird unter die obersten `k` Pfannkuchen geschoben, diese `k` werden **umgedreht**, und dann wird der **oberste** Pfannkuchen **gegessen**. Der Stapel hat danach einen Pfannkuchen weniger. Ziel ist **irgendein** sortierter Stapel (es ist egal, wie viele übrig bleiben). Gesucht ist die **kleinste** Zahl von Operationen A(S) für einen gegebenen Stapel S.

Beispiel: Aus `(3, 2, 4, 5, 1)` (von oben nach unten) wird mit dem Wender zwischen 4 und 5 (also `k = 3`) der Stapel `(2, 3, 5, 1)`.

Die Dateien `daten/pancake_mini0.txt` bis `pancake_mini4.txt` enthalten Stapel mit 5, 7, 8, 11 und 13 Pfannkuchen (erste Zahl: wie viele, danach die Größen von oben nach unten); es sind die ersten fünf Beispiele der Aufgabe.

In [ ]:
def lies_stapel(dateiname):
    """Erste Zahl: Anzahl n, danach die Größen der Pfannkuchen von oben nach unten."""
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(x) for x in datei.read().split()]
    assert zahlen[0] == len(zahlen) - 1
    return tuple(zahlen[1:])


def ist_sortiert(stapel):
    for i in range(len(stapel) - 1):
        if stapel[i] > stapel[i + 1]:
            return False
    return True


def mit_leerzeichen(zahl):
    return '{:,}'.format(zahl).replace(',', ' ')


P_DATEIEN = ['daten/pancake_mini%d.txt' % i for i in range(5)]
P_STAPEL = [lies_stapel(d) for d in P_DATEIEN]
for stapel in P_STAPEL:
    print(len(stapel), 'Pfannkuchen:', stapel)

**Normieren** (siehe Abschnitt 1): Ob der Stapel sortiert ist und wie viele Operationen er braucht, hängt nur von der **Reihenfolge** der Größen ab, nicht von den Zahlen selbst. `(2, 3, 5, 1)` und `(2, 3, 4, 1)` sind dasselbe Problem. Man nummeriert nach jeder Operation die Größen wieder von 1 bis zur neuen Länge durch. Aus 5 Pfannkuchen werden sonst nach ein paar Operationen viele Zustände, die nur anders beschriftet sind.

#### Aufgabe E1.1

Schreibe `wende_und_iss(stapel, k)` und `normiere(stapel)`. Die Stapel sind **Tupel** (damit sie Schlüssel eines Wörterbuchs sein können). Mit `stapel[k - 1::-1]` bekommst du die obersten `k` Einträge rückwärts.

**Erwartete Ausgabe:**
```
(2, 3, 5, 1) (2, 3, 4, 1)
(2, 4, 5)
```

In [ ]:
def wende_und_iss(stapel, k):
    """Den Wender unter die obersten k Pfannkuchen schieben, umdrehen, den obersten essen. Gibt den neuen Stapel zurück."""
    # TODO
    return stapel


def normiere(stapel):
    """Die Größen wieder von 1 bis m durchnummerieren, die Reihenfolge bleibt."""
    # TODO
    return stapel


s = (3, 2, 4, 5, 1)
print(wende_und_iss(s, 3), normiere(wende_und_iss(s, 3)))
print(wende_und_iss(wende_und_iss(s, 5), 4))

<details>
<summary>Tipp anzeigen</summary>

`wende_und_iss`: `gedreht = stapel[k - 1::-1] + stapel[k:]` (die obersten `k` rückwärts, dann der Rest), dann `gedreht[1:]`. `normiere`: ein Wörterbuch `rang`, das jeder Größe ihren Platz in `sorted(stapel)` (ab 1) zuordnet.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def wende_und_iss(stapel, k):
    """Den Wender unter die obersten k Pfannkuchen schieben, umdrehen, den obersten essen. Gibt den neuen Stapel zurück."""
    gedreht = stapel[k - 1::-1] + stapel[k:]          # die obersten k umdrehen (Tupel rückwärts zerschneiden)
    return gedreht[1:]                                # den obersten essen


def normiere(stapel):
    """Die Größen wieder von 1 bis m durchnummerieren, die Reihenfolge bleibt."""
    rang = {}
    for platz, groesse in enumerate(sorted(stapel)):
        rang[groesse] = platz + 1
    return tuple(rang[g] for g in stapel)


s = (3, 2, 4, 5, 1)
print(wende_und_iss(s, 3), normiere(wende_und_iss(s, 3)))
print(wende_und_iss(wende_und_iss(s, 5), 4))
```

</details>

Jetzt ein **allgemeines** Gerüst: Du kennst `breitensuche` und `astern` aus den Abschnitten 2 und 4, hier mit **zwei Funktionen mehr als Argument**: `nachfolger(zustand)` liefert die Folgezustände, `ist_ziel(zustand)` sagt, ob das Ziel erreicht ist. Dann gelten sie für **jede** Aufgabe mit Zuständen. Zusätzlich merken sie sich zu jedem Zustand den Vorgänger und geben den **ganzen Weg** zurück.

In [ ]:
def breitensuche_allg(start, nachfolger, ist_ziel):
    """Wie breitensuche oben, aber für jede Aufgabe: nachfolger(zustand) liefert die Liste der Folgezustände,
    ist_ziel(zustand) sagt, ob das Ziel erreicht ist. Gibt (Weg als Liste von Zuständen, bearbeitete Zustände) zurück."""
    vorgaenger = {start: None}
    schlange = deque([start])
    bearbeitet = 0
    while schlange:
        zustand = schlange.popleft()
        bearbeitet += 1
        if ist_ziel(zustand):
            return weg_zurueck(vorgaenger, zustand), bearbeitet
        for neu in nachfolger(zustand):
            if neu not in vorgaenger:
                vorgaenger[neu] = zustand
                schlange.append(neu)
    return None, bearbeitet


def astern_allg(start, nachfolger, ist_ziel, schaetzung):
    """Wie astern oben, aber für jede Aufgabe. Gibt (Weg als Liste von Zuständen, bearbeitete Zustände) zurück."""
    beste = {start: 0}
    vorgaenger = {start: None}
    warteschlange = [(schaetzung(start), 0, start)]
    bearbeitet = 0
    while warteschlange:
        f, g, zustand = heapq.heappop(warteschlange)
        if g > beste[zustand]:
            continue
        bearbeitet += 1
        if ist_ziel(zustand):
            return weg_zurueck(vorgaenger, zustand), bearbeitet
        for neu in nachfolger(zustand):
            if neu not in beste or g + 1 < beste[neu]:
                beste[neu] = g + 1
                vorgaenger[neu] = zustand
                heapq.heappush(warteschlange, (g + 1 + schaetzung(neu), g + 1, neu))
    return None, bearbeitet


def weg_zurueck(vorgaenger, zustand):
    """Der Weg vom Start bis zustand (über die Vorgänger rückwärts gesammelt und umgedreht)."""
    weg = [zustand]
    while vorgaenger[zustand] is not None:
        zustand = vorgaenger[zustand]
        weg.append(zustand)
    weg.reverse()
    return weg

Jetzt die Breitensuche mit **normierten** Stapeln. Die Funktion `nachfolger_pancake` gibt zu jedem Stapel die normierten Stapel nach den Operationen `k = 1` bis zur Länge zurück. Das Ziel ist `ist_sortiert`.

In [ ]:
def nachfolger_pancake(stapel):
    """Alle normierten Stapel, die man mit einer Wende-und-Ess-Operation erreicht (k = 1 bis Länge)."""
    liste = []
    for k in range(1, len(stapel) + 1):
        liste.append(normiere(wende_und_iss(stapel, k)))
    return liste


p_bfs = []
for stapel in P_STAPEL:
    weg, bearbeitet = breitensuche_allg(normiere(stapel), nachfolger_pancake, ist_sortiert)
    p_bfs.append((len(weg) - 1, bearbeitet))
    print('%2d Pfannkuchen: A(S) = %d, bearbeitete Stapel: %s' % (len(stapel), len(weg) - 1, mit_leerzeichen(bearbeitet)))

 5 Pfannkuchen: A(S) = 2, bearbeitete Stapel: 9
 7 Pfannkuchen: A(S) = 3, bearbeitete Stapel: 59
 8 Pfannkuchen: A(S) = 4, bearbeitete Stapel: 142
11 Pfannkuchen: A(S) = 6, bearbeitete Stapel: 3 034
13 Pfannkuchen: A(S) = 7, bearbeitete Stapel: 26 242


Die Zahl der bearbeiteten Stapel wächst schnell: von 9 bei 5 Pfannkuchen auf 26 242 bei 13. (A(S) ist 2, 3, 4, 6 und 7: Das stimmt mit dem Lösungsnotebook überein.)

**Eine Schätzung: Breakpoints.** Ein **Breakpoint** ist eine Stelle, an der zwei benachbarte Pfannkuchen **keine benachbarten Größen** haben (|a − b| ≠ 1), dazu zählt das untere Ende, wenn der unterste nicht der größte ist. Der Stapel `(1, 2, 3, 4)` hat keinen, `(1, 2, 4, 3)` hat zwei (zwischen 2 und 4, und unten, weil 3 nicht der größte ist). Ein sortierter Stapel hat 0 Breakpoints. Eine Operation beseitigt **höchstens 3** Breakpoints (das behaupten die offiziellen Lösungshinweise mit einem Beweis; wir prüfen es unten mit einer Rechnung). Also braucht man mindestens **Breakpoints ÷ 3** Operationen (aufgerundet): eine **zulässige** Schätzung.

#### Aufgabe E1.2

Schreibe `breakpoints(stapel)` und `schaetzung_pancake(stapel)` (aufgerundet: `-(-b // 3)`). Das Gerüst druckt dir die Schätzungen für die fünf Stapel aus der Datei.

**Erwartete Ausgabe:**
```
2 0 3
[1, 3, 3, 4, 4]
```

In [ ]:
def breakpoints(stapel):
    """Anzahl der Stellen, an denen zwei Nachbarn keine benachbarten Größen haben (|a - b| != 1),
    dazu 1, wenn der unterste Pfannkuchen nicht der größte ist."""
    # TODO
    return 0


def schaetzung_pancake(stapel):
    """Eine Operation beseitigt höchstens 3 Breakpoints. Also braucht man mindestens breakpoints / 3 Operationen (aufgerundet)."""
    # TODO
    return 0


print(breakpoints((1, 2, 4, 3)), breakpoints((1, 2, 3, 4)), breakpoints((3, 2, 4, 5, 1)))
print([schaetzung_pancake(s) for s in P_STAPEL])

<details>
<summary>Tipp anzeigen</summary>

In `breakpoints`: eine Schleife über die Paare von Nachbarn plus ein Test für das untere Ende (`stapel[-1] != len(stapel)`). Aufrunden von `a / 3` ohne `math`: `-(-a // 3)` (Ganzzahldivision rundet nach unten, mit zwei Minuszeichen daraus nach oben).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def breakpoints(stapel):
    """Anzahl der Stellen, an denen zwei Nachbarn keine benachbarten Größen haben (|a - b| != 1),
    dazu 1, wenn der unterste Pfannkuchen nicht der größte ist."""
    anzahl = 0
    for i in range(len(stapel) - 1):
        if abs(stapel[i] - stapel[i + 1]) != 1:
            anzahl += 1
    if stapel[-1] != len(stapel):
        anzahl += 1
    return anzahl


def schaetzung_pancake(stapel):
    """Eine Operation beseitigt höchstens 3 Breakpoints. Also braucht man mindestens breakpoints / 3 Operationen (aufgerundet)."""
    return -(-breakpoints(stapel) // 3)           # aufrunden ohne math: -(-a // b)


print(breakpoints((1, 2, 4, 3)), breakpoints((1, 2, 3, 4)), breakpoints((3, 2, 4, 5, 1)))
print([schaetzung_pancake(s) for s in P_STAPEL])
```

</details>

In [ ]:
print('Stapel         A(S)   BFS bearbeitet   A* bearbeitet')
p_astern = []
for i, stapel in enumerate(P_STAPEL):
    weg, bearbeitet = astern_allg(normiere(stapel), nachfolger_pancake, ist_sortiert, schaetzung_pancake)
    assert len(weg) - 1 == p_bfs[i][0]                     # beide Verfahren: gleich viele Operationen
    p_astern.append((len(weg) - 1, bearbeitet))
    print('%2d Pfannkuchen %5d %14s %15s' % (len(stapel), len(weg) - 1, mit_leerzeichen(p_bfs[i][1]), mit_leerzeichen(bearbeitet)))

Stapel         A(S)   BFS bearbeitet   A* bearbeitet
 5 Pfannkuchen     2              9               5
 7 Pfannkuchen     3             59              19
 8 Pfannkuchen     4            142              83
11 Pfannkuchen     6          3 034           2 072
13 Pfannkuchen     7         26 242          14 020


**A\* bearbeitet weniger Stapel als die Breitensuche** (bei 11 Pfannkuchen 2 072 statt 3 034, bei 13 14 020 statt 26 242), und die Zahlen für A\* stimmen mit denen im [Lösungsnotebook](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/41_PancakeSort/pancake_loesung.ipynb) überein (dessen Breitensuche zählt die gesehenen statt der bearbeiteten Stapel, deshalb weichen diese Zahlen leicht ab). Der Gewinn ist hier kleiner als beim Schiebepuzzle: Die Breakpoint-Schätzung ist **grob** (Division durch 3).

**Die Schätzung prüfen.** Die Behauptung „höchstens 3 Breakpoints pro Operation“ stammt aus den Lösungshinweisen. Wir verlassen uns nicht darauf, sondern berechnen für **alle** Stapel bis 7 Pfannkuchen die wahre Zahl A(S) (Rekursion: A(S) = 1 + das Kleinste der A-Werte der Nachfolger) und prüfen, ob die Schätzung nie zu groß ist. Nebenbei ergibt das die größten Werte P(n) = größtes A(S) bei n Pfannkuchen, die in der Aufgabe stehen (P(1) bis P(7) = 0, 1, 2, 2, 3, 3, 4).

In [ ]:
# Für ALLE Stapel bis 7 Pfannkuchen die wahre Zahl A(S) ausrechnen: A(S) = 1 + kleinstes A der n Nachfolger.
import itertools

wahr = {}
p_max = {}
for n in range(1, 8):
    p_max[n] = 0
    for stapel in itertools.permutations(range(1, n + 1)):
        if ist_sortiert(stapel):
            wahr[stapel] = 0
        else:
            wahr[stapel] = 1 + min(wahr[f] for f in nachfolger_pancake(stapel))
        p_max[n] = max(p_max[n], wahr[stapel])

zu_gross = 0
for stapel in wahr:
    if schaetzung_pancake(stapel) > wahr[stapel]:
        zu_gross += 1
print('Stapel geprüft:', mit_leerzeichen(len(wahr)), '  Schätzung größer als A(S):', zu_gross)
print('P(n) für n = 1 bis 7:', [p_max[n] for n in range(1, 8)])

Stapel geprüft: 5 913   Schätzung größer als A(S): 0
P(n) für n = 1 bis 7: [0, 1, 2, 2, 3, 3, 4]


#### Frage E1.3

1. In der Ausgabe oben steht „Schätzung größer als A(S): 0“ für 5 913 Stapel. Beweist das, dass die Breakpoint-Schätzung für **alle** Stapel zulässig ist (auch bei 13 Pfannkuchen)?
2. Warum findet A\* mit dieser Schätzung trotzdem den **kürzesten** Weg, obwohl sie meist weit unter der Wahrheit liegt?
3. Für P(8) bis P(11) (Teil b der Aufgabe) müsste man A(S) für **alle** Stapel mit 8 bis 11 Pfannkuchen kennen. Warum reicht „A\* für jeden Stapel einzeln“ nicht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. **Nein.** Es ist ein **Test** für kleine Fälle, kein Beweis. Er stützt die Behauptung aus den Lösungshinweisen (mit Beweis), sodass wir ihr trauen. Bei einer eigenen Schätzung brauchst du ein Argument wie „jede Operation ändert höchstens …“, nicht nur Tests. (Und es lohnt sich, solche Tests trotzdem zu machen: Sie finden Denkfehler im Argument.)
2. Es kommt nur darauf an, dass sie **nie zu groß** ist. Eine zu kleine Schätzung macht A\* nur langsamer (mehr Zustände), nicht falsch.
3. Es gibt 8! = 40 320 bis 11! = 39 916 800 Stapel. Das ist zu viel für einzelne Suchen. Das [Lösungsnotebook](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/41_PancakeSort/pancake_loesung.ipynb) rechnet **rückwärts** die Menge aller Stapel aus, die mindestens `d` Operationen brauchen, und kommt so bis P(10) exakt (und mit A\*-Beispielen als Untergrenze bis P(15)). Das ist die Idee „vom Ziel aus rechnen“ aus Abschnitt 2.

</details>

## Wettbewerbs-Ecke 2: Simultane Labyrinthe

*Angelehnt an Aufgabe 2 „Simultane Labyrinthe“ der 2. Runde des 43. Bundeswettbewerbs Informatik (2024/25).*

Zwei Labyrinthe (Gitter gleicher Größe) mit Wänden und manchmal **Gruben**. Zwei Personen starten **gleichzeitig** auf dem Feld oben links, eine in jedem Labyrinth, und wollen zum Feld unten rechts. Sie bekommen **dieselbe Folge von Anweisungen** (hoch, rechts, runter, links): Bei jeder Anweisung geht **jede** Person ein Feld in diese Richtung. Läuft sie gegen eine Wand (oder aus dem Gitter), bleibt sie stehen. Wer das Ziel erreicht hat, bleibt dort. Fällt jemand in eine **Grube**, geht er zurück zum Start. Gesucht ist eine **kürzeste** Folge, mit der **beide** am Ziel sind.

Die Dateien `daten/labyrinthe_mini0.txt` bis `labyrinthe_mini2.txt` sind die ersten drei Beispiele der Aufgabe (3 × 3, 5 × 4 und 10 × 10 Felder; Nr. 0 ist das Beispiel aus der Aufgabenstellung, Nr. 2 hat Gruben). Der **Zustand** ist das **Paar** der beiden Felder (a, b): Bei 10 × 10 Feldern sind das 100² = 10 000 Zustände, bei 250 × 250 Feldern 3,9 Milliarden. Vom Zustand zu den vier Folgezuständen führen die **Zugtabellen** (eine je Labyrinth): Wände, Ziel und Gruben stecken schon in der Tabelle, die Anweisung 0 bis 3 wirkt für jedes Feld wie ein Nachschlagen. Den Code zum Einlesen und zum Bauen der Tabellen bekommst du fertig.

In [ ]:
PFEILE = '↑→↓←'
RICHTUNGEN = [(0, -1), (1, 0), (0, 1), (-1, 0)]          # (dx, dy) zu ↑ → ↓ ←


def lies_labyrinthe(dateiname):
    """Gibt n, m und eine Liste der beiden Labyrinthe zurück; jedes ist (senkrecht, waagerecht, gruben)."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = [z.split() for z in datei.read().splitlines()]
    n, m = int(zeilen[0][0]), int(zeilen[0][1])
    zeile = 1
    labyrinthe = []
    for _ in range(2):
        senkrecht = []
        for y in range(m):
            senkrecht.append([int(x) for x in zeilen[zeile]])
            zeile += 1
        waagerecht = []
        for y in range(m - 1):
            waagerecht.append([int(x) for x in zeilen[zeile]])
            zeile += 1
        anzahl_gruben = int(zeilen[zeile][0])
        zeile += 1
        gruben = []
        for _ in range(anzahl_gruben):
            gruben.append((int(zeilen[zeile][0]), int(zeilen[zeile][1])))
            zeile += 1
        labyrinthe.append((senkrecht, waagerecht, gruben))
    return n, m, labyrinthe


def zugtabelle(n, m, labyrinth):
    """tabelle[r][feld] ist das Feld, auf dem man nach der Anweisung r (0 = hoch, 1 = rechts, 2 = runter, 3 = links)
    steht, wenn man auf feld = y * n + x stand. Wand: bleibt stehen. Ziel: bleibt stehen. Grube: zurück auf Feld 0.
    Auf dem Grubenfeld selbst steht man nie (None)."""
    senkrecht, waagerecht, gruben = labyrinth
    ziel = n * m - 1
    grubenfelder = set()
    for x, y in gruben:
        grubenfelder.add(y * n + x)
    tabelle = []
    for r in range(4):
        dx, dy = RICHTUNGEN[r]
        zeile = []
        for feld in range(n * m):
            x, y = feld % n, feld // n
            if feld in grubenfelder:
                neu = None
            elif feld == ziel:
                neu = feld
            else:
                nx, ny = x + dx, y + dy
                if nx < 0 or nx >= n or ny < 0 or ny >= m:
                    neu = feld
                elif r == 1 and senkrecht[y][x] == 1:
                    neu = feld
                elif r == 3 and senkrecht[y][x - 1] == 1:
                    neu = feld
                elif r == 2 and waagerecht[y][x] == 1:
                    neu = feld
                elif r == 0 and waagerecht[y - 1][x] == 1:
                    neu = feld
                else:
                    neu = ny * n + nx
                    if neu in grubenfelder:
                        neu = 0
            zeile.append(neu)
        tabelle.append(zeile)
    return tabelle


def lade_labyrinthe(nr):
    n, m, labyrinthe = lies_labyrinthe('daten/labyrinthe_mini%d.txt' % nr)
    return n, m, zugtabelle(n, m, labyrinthe[0]), zugtabelle(n, m, labyrinthe[1])


n_lab, m_lab, tab1, tab2 = lade_labyrinthe(0)
print(n_lab, 'x', m_lab, ' Ziel ist Feld', n_lab * m_lab - 1)
print('1. Labyrinth, Anweisung → von Feld 0 bis 8:', tab1[1])
print('2. Labyrinth, Anweisung ↓ von Feld 0 bis 8:', tab2[2], ' (Feld 6 ist eine Grube)')

#### Aufgabe E2.1

Schreibe `nachfolger_paar(zustand)` (die vier Folgezustände: für jede Anweisung `r` das Paar `(tab1[r][a], tab2[r][b])`) und `ist_ziel_paar(zustand)`. Danach läuft die allgemeine Breitensuche aus der vorigen Ecke. Die Ausgabe zeigt, wie viele Anweisungen nötig sind und wie viele Zustände bearbeitet wurden.

**Erwartete Ausgabe:**
```
labyrinthe_mini0: (8, 29)
labyrinthe_mini1: (31, 341)
labyrinthe_mini2: (65, 8189)
```

In [ ]:
def nachfolger_paar(zustand):
    """zustand ist das Paar (a, b) der Felder der beiden Personen. Gibt die Liste der vier Folgezustände zurück
    (je einer für die Anweisungen 0 bis 3). Benutzt die Tabellen tab1 und tab2."""
    # TODO
    return []


def ist_ziel_paar(zustand):
    """Stehen beide auf dem Zielfeld? Das Zielfeld ist das letzte Feld: len(tab1[0]) - 1."""
    # TODO
    return False


def zeige_labyrinth_ergebnis(nr):
    global tab1, tab2
    n, m, tab1, tab2 = lade_labyrinthe(nr)
    weg, bearbeitet = breitensuche_allg((0, 0), nachfolger_paar, ist_ziel_paar)
    if weg is None:
        return None, bearbeitet
    return len(weg) - 1, bearbeitet


for nr in range(3):
    print('labyrinthe_mini%d:' % nr, zeige_labyrinth_ergebnis(nr))

<details>
<summary>Tipp anzeigen</summary>

`nachfolger_paar`: `a, b = zustand`, eine Schleife über `r in range(4)` und `(tab1[r][a], tab2[r][b])` anhängen. `ist_ziel_paar`: das Zielfeld ist `len(tab1[0]) - 1`, der Zustand ist das Ziel, wenn er `(ziel, ziel)` ist.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def nachfolger_paar(zustand):
    """zustand ist das Paar (a, b) der Felder der beiden Personen. Gibt die Liste der vier Folgezustände zurück
    (je einer für die Anweisungen 0 bis 3). Benutzt die Tabellen tab1 und tab2."""
    a, b = zustand
    liste = []
    for r in range(4):
        liste.append((tab1[r][a], tab2[r][b]))
    return liste


def ist_ziel_paar(zustand):
    """Stehen beide auf dem Zielfeld? Das Zielfeld ist das letzte Feld: len(tab1[0]) - 1."""
    ziel = len(tab1[0]) - 1
    return zustand == (ziel, ziel)


def zeige_labyrinth_ergebnis(nr):
    global tab1, tab2
    n, m, tab1, tab2 = lade_labyrinthe(nr)
    weg, bearbeitet = breitensuche_allg((0, 0), nachfolger_paar, ist_ziel_paar)
    if weg is None:
        return None, bearbeitet
    return len(weg) - 1, bearbeitet


for nr in range(3):
    print('labyrinthe_mini%d:' % nr, zeige_labyrinth_ergebnis(nr))
```

</details>

Die Längen 8, 31 und 65 stimmen mit dem [Lösungsnotebook](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/43_SimultaneLabyrinthe/simultanelabyrinthe_loesung.ipynb) überein (bei Nr. 0 braucht die Folge aus der Aufgabenstellung 9 Anweisungen, die kürzeste hat 8). Bei 10 × 10 Feldern bearbeitet die Breitensuche 8 189 von höchstens 10 000 Zuständen.

**Die Schätzung: Abstände vom Ziel aus.** Jede Person braucht **mindestens** so viele Anweisungen, wie ihr Feld im eigenen Labyrinth vom Ziel entfernt ist, **unabhängig von der anderen**. Beide bekommen dieselben Anweisungen, also braucht man **mindestens das Maximum** der beiden Abstände. Die Abstände aller Felder zum Ziel rechnet man **einmal** aus, mit einer Breitensuche **rückwärts vom Ziel** (Abschnitt 2): Man braucht für jedes Feld die Liste der Felder, von denen man mit **einer** Anweisung dorthin kommt (`davor`). Die Züge sind hier **nicht umkehrbar** (an einer Wand bleibt man stehen), deshalb baut man `davor` aus der Zugtabelle auf. Felder, von denen man das Ziel nie erreicht, behalten den Abstand −1.

#### Aufgabe E2.2

Vervollständige `distanzen(tab)`: Eine Breitensuche vom Zielfeld aus über die Listen `davor`. Ein Feld bekommt den Abstand des Feldes, von dem aus du es entdeckst, plus 1, aber nur, wenn sein Abstand noch `-1` ist. Das Gerüst zeigt die Abstände für beide Labyrinthe aus der Aufgabenstellung.

**Erwartete Ausgabe:**
```
  8   3   2
  7   4   1
  6   5   0

  6   3   2
  5   4   1
 -1   1   0
```

In [ ]:
def distanzen(tab):
    """Zu jedem Feld die kleinste Zahl von Anweisungen bis zum Zielfeld (-1, wenn man es nie erreicht).
    Rückwärts vom Ziel aus: davor[feld] ist die Liste der Felder, von denen man mit EINER Anweisung auf feld kommt."""
    felder = len(tab[0])
    ziel = felder - 1
    davor = [[] for _ in range(felder)]
    for r in range(4):
        for feld in range(felder):
            neu = tab[r][feld]
            if neu is not None and neu != feld:
                davor[neu].append(feld)
    dist = [-1] * felder
    dist[ziel] = 0
    # TODO: Breitensuche vom Ziel aus über davor. Ein Feld, dessen Abstand noch -1 ist, bekommt den Abstand + 1.
    return dist


def zeige_abstaende(dist, n):
    for y in range(len(dist) // n):
        print(' '.join('%3d' % d for d in dist[y * n:(y + 1) * n]))


n_lab, m_lab, tab1, tab2 = lade_labyrinthe(0)
zeige_abstaende(distanzen(tab1), n_lab)
print()
zeige_abstaende(distanzen(tab2), n_lab)

<details>
<summary>Tipp anzeigen</summary>

Wie jede Breitensuche mit `deque`: `feld = schlange.popleft()`, dann für jedes `vorher in davor[feld]`: wenn `dist[vorher] == -1`, setze `dist[vorher] = dist[feld] + 1` und stelle `vorher` in die Schlange.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def distanzen(tab):
    """Zu jedem Feld die kleinste Zahl von Anweisungen bis zum Zielfeld (-1, wenn man es nie erreicht).
    Rückwärts vom Ziel aus: davor[feld] ist die Liste der Felder, von denen man mit EINER Anweisung auf feld kommt."""
    felder = len(tab[0])
    ziel = felder - 1
    davor = [[] for _ in range(felder)]
    for r in range(4):
        for feld in range(felder):
            neu = tab[r][feld]
            if neu is not None and neu != feld:
                davor[neu].append(feld)
    dist = [-1] * felder
    dist[ziel] = 0
    schlange = deque([ziel])
    while schlange:
        feld = schlange.popleft()
        for vorher in davor[feld]:
            if dist[vorher] == -1:
                dist[vorher] = dist[feld] + 1
                schlange.append(vorher)
    return dist


def zeige_abstaende(dist, n):
    for y in range(len(dist) // n):
        print(' '.join('%3d' % d for d in dist[y * n:(y + 1) * n]))


n_lab, m_lab, tab1, tab2 = lade_labyrinthe(0)
zeige_abstaende(distanzen(tab1), n_lab)
print()
zeige_abstaende(distanzen(tab2), n_lab)
```

</details>

Im 2. Labyrinth ist das Feld unten links die Grube (Abstand −1: Man steht nie dort). Der Abstand des Startfelds ist 8 im ersten und 6 im zweiten Labyrinth: Man braucht also **mindestens 8** Anweisungen, und es sind genau 8.

#### Aufgabe E2.3

Schreibe die Schätzung `schaetzung_paar(zustand)` in `labyrinth_astern` und starte A\* mit `astern_allg`. Die Funktion `schaetzung_paar` ist **innerhalb** von `labyrinth_astern` definiert, damit sie `dist1` und `dist2` kennt. Vergleiche die Zahl der bearbeiteten Zustände mit der Breitensuche aus E2.1.

**Erwartete Ausgabe:**
```
labyrinthe_mini0: (8, 9)
labyrinthe_mini1: (31, 150)
labyrinthe_mini2: (65, 3012)
```

In [ ]:
def labyrinth_astern(nr):
    """Löst labyrinthe_mini<nr> mit A*. Gibt (Zahl der Anweisungen, bearbeitete Zustände) zurück."""
    global tab1, tab2
    n, m, tab1, tab2 = lade_labyrinthe(nr)
    dist1 = distanzen(tab1)
    dist2 = distanzen(tab2)

    def schaetzung_paar(zustand):
        # TODO: Jede Person braucht mindestens so viele Anweisungen wie ihr Abstand zum Ziel. Beide bekommen
        # dieselben Anweisungen, also braucht man mindestens ... (der größere der beiden Abstände).
        return 0

    weg, bearbeitet = astern_allg((0, 0), nachfolger_paar, ist_ziel_paar, schaetzung_paar)
    return len(weg) - 1, bearbeitet


for nr in range(3):
    print('labyrinthe_mini%d:' % nr, labyrinth_astern(nr))

<details>
<summary>Tipp anzeigen</summary>

Der Zustand ist das Paar `(a, b)`; die Schätzung ist `max(dist1[zustand[0]], dist2[zustand[1]])`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def labyrinth_astern(nr):
    """Löst labyrinthe_mini<nr> mit A*. Gibt (Zahl der Anweisungen, bearbeitete Zustände) zurück."""
    global tab1, tab2
    n, m, tab1, tab2 = lade_labyrinthe(nr)
    dist1 = distanzen(tab1)
    dist2 = distanzen(tab2)

    def schaetzung_paar(zustand):
        # Jede Person braucht mindestens so viele Anweisungen wie ihr Abstand zum Ziel. Beide bekommen
        # dieselben Anweisungen, also braucht man mindestens den größeren der beiden Abstände.
        return max(dist1[zustand[0]], dist2[zustand[1]])

    weg, bearbeitet = astern_allg((0, 0), nachfolger_paar, ist_ziel_paar, schaetzung_paar)
    return len(weg) - 1, bearbeitet


for nr in range(3):
    print('labyrinthe_mini%d:' % nr, labyrinth_astern(nr))
```

</details>

**Die Tabelle:**

| Beispiel | Anweisungen | Breitensuche | A\* mit Abständen |
|---|---|---|---|
| `labyrinthe_mini0` (3 × 3) | 8 | 29 | 9 |
| `labyrinthe_mini1` (5 × 4) | 31 | 341 | 150 |
| `labyrinthe_mini2` (10 × 10) | 65 | 8 189 | 3 012 |

(bearbeitete Zustände). A\* bearbeitet **weniger als die Hälfte**; die Zahlen für A\* sind dieselben wie im Lösungsnotebook (dessen Breitensuche zählt gesehene Zustände: 32, 342 und 8 206). Im Lösungsnotebook läuft diese Idee bis zu Labyrinthen mit 250 × 250 Feldern (rund 3,9 Milliarden mögliche Zustände) und bearbeitet dort etwa 485 000 Zustände: Für das Paar-Problem ist die **einzeln berechnete** Schätzung sehr stark.

#### Frage E2.4

1. Warum ist `max(dist1[a], dist2[b])` **zulässig**? Warum nicht die **Summe** der beiden Abstände?
2. Die Gruben setzen eine Person zum Start zurück. Warum bleibt die Schätzung zulässig? (Was bedeutet das für `davor` und die Abstände?)
3. Wie erkennt man **unlösbare** Labyrinthpaare sofort, ohne eine Suche über Paare?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jede Anweisung geht für beide gleichzeitig. Wenn die eine Person mindestens `x` Anweisungen braucht und die andere mindestens `y`, dann braucht die Folge mindestens `max(x, y)` Anweisungen: Beide laufen **parallel**, die Anweisungen werden nicht addiert. Die Summe wäre zu groß (Chris' Idee „erst die eine, dann die andere“ hätte 12 Anweisungen gebraucht, die kürzeste Folge im Beispiel hat 8): Mit der Summe wäre die Schätzung nicht zulässig, und A\* könnte das Ergebnis verfehlen.
2. Die Tabelle `tab` enthält die Gruben als „Rücksprung nach Feld 0“; `distanzen` benutzt dieselbe Tabelle rückwärts. Der Abstand ist also der **wahre** kürzeste Abstand in diesem Labyrinth **mit** den Gruben. Eine Person braucht diese Zahl von Anweisungen mindestens, auch wenn die andere schon am Ziel ist.
3. Ist im Labyrinth 1 oder im Labyrinth 2 der Abstand des Startfelds **−1**, kann diese Person das Ziel allein nie erreichen, also auch nicht zu zweit. Das ist eine **Schranke aus der Struktur** (wie die Parität beim Schiebepuzzle), sie kostet nur zwei einfache Breitensuchen über **Felder** statt über Paare. Umgekehrt beweist das „ja“ nicht, dass es eine gemeinsame Folge gibt; wie das Lösungsnotebook dazu argumentiert, steht dort.

</details>

## Für Schnelle: Schach dem Wildschwein (nur als Verweis)

*Angelehnt an Aufgabe 3 „Schach dem Wildschwein“ der 2. Runde des 37. Bundeswettbewerbs Informatik (2018/19).*

Auch diese Aufgabe ist eine Suche in einem Zustandsraum, aber mit **zwei Gegnern**: Der Jäger (mit den Hunden) will das Wildschwein auf ein Zielfeld zwingen, das Wildschwein will das verhindern. Ein Zustand ist eine **Stellung** (Felder der Figuren, wer am Zug ist). Drei Gedanken aus diesem Notebook kommen wieder vor:

- **Tiefenschranke:** „Kann der Jäger in höchstens `n` Zügen gewinnen?“ lässt sich rekursiv beantworten, indem man die Schranke bei jedem Zug um 1 senkt. Hat der Jäger den Zug, gewinnt er, wenn **einer** seiner Züge zu einer Gewinnstellung führt; hat das Wildschwein den Zug, gewinnt der Jäger nur, wenn **alle** Antworten ihm helfen.
- **Vom Ziel aus rechnen (Rückwärtsanalyse):** Man beginnt bei den **Endstellungen** und rechnet Schicht für Schicht zurück, welche Stellungen in 1, 2, 3 … Zügen gewonnen sind. Das ist die Breitensuche vom Ziel aus aus Abschnitt 2, nur mit der „einer / alle“-Regel statt „kleinster Abstand“.
- **Zustände kompakt nummerieren:** Bei Millionen Stellungen speichert man jede Stellung als **Nummer** in einer langen Liste.

Das Lösungsnotebook [Schach dem Wildschwein](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/37_SchachDemWildschwein/schach_loesung.ipynb) zeigt die einfache Suche mit Tiefenschranke und die Rückwärtsanalyse. Es ist deutlich länger als die beiden Ecken oben; bearbeite es erst, wenn du die beiden Ecken gelöst hast.

## Zum Schluss: Das Schiebepuzzle mit MiniZinc

Man kann die **Frage nach einer Folge von genau `T` Zügen** auch als Constraint-Problem aufschreiben. Die Zahl der Züge `T` ist **fest**: Der Solver soll für jeden Zeitpunkt `t = 0..T` den Zustand des Bretts bestimmen, so dass der Anfang der Start ist, das Ende das Ziel, und jeder Schritt ein erlaubter Zug. Das ist eine **Planungsaufgabe mit fester Länge**.

Aus den Arbeitsblättern zu MiniZinc kennst du `array`, `forall`, `alldifferent` und Variablen als Index. Neu ist nur:

- `b[t, f]` ist **ein zweidimensionales Feld von Variablen** (Zeit × Feld), `p[t]` das Feld der Lücke zur Zeit `t`,
- `nachbar[p[t], p[t + 1]]` benutzt **zwei Variablen als Index** in ein Datenfeld (die Lücke darf nur auf ein Nachbarfeld wandern),
- `if … then … elseif … else … endif` innerhalb einer Bedingung beschreibt, was ein Zug mit jedem Feld macht.

Das Modell läuft **nicht in JupyterLite**. Kopiere es in die MiniZinc IDE oder in den [MiniZinc Playground](https://play.minizinc.org/); die Daten (`T`, `start`, `nachbar`) erzeugt die Python-Zelle darunter als `.dzn`-Datei.

```minizinc
include "alldifferent.mzn";

int: T;                                          % genau so viele Züge
array[1..9] of int: start;                       % Startzustand, 0 ist die Lücke
array[1..9, 1..9] of bool: nachbar;              % nachbar[p, q]: sind die Felder p und q benachbart?

array[0..T, 1..9] of var 0..8: b;                % b[t, f]: Kachel auf Feld f nach t Zügen
array[0..T] of var 1..9: p;                      % p[t]: Feld der Lücke nach t Zügen

constraint forall(f in 1..9)(b[0, f] = start[f]);
constraint forall(f in 1..8)(b[T, f] = f) /\ b[T, 9] = 0;          % am Ende: das Ziel
constraint forall(t in 0..T)(b[t, p[t]] = 0);                      % die Lücke steht auf Feld p[t]
constraint forall(t in 0..T)(alldifferent([b[t, f] | f in 1..9])); % jede Kachel genau einmal
% Ein Zug: die Lücke wandert auf ein Nachbarfeld, dessen Kachel rutscht auf das alte Lückenfeld.
constraint forall(t in 0..T-1)(nachbar[p[t], p[t + 1]]);
constraint forall(t in 0..T-1, f in 1..9)(
  b[t + 1, f] = if f = p[t] then b[t, p[t + 1]]
                elseif f = p[t + 1] then 0
                else b[t, f] endif);

solve satisfy;
output ["p = ", show(p), "\n"];
```

In [ ]:
def als_dzn(zustand, zuege):
    """Die Datendatei für das MiniZinc-Modell: Zahl der Züge, Startzustand und die Nachbarschaft der Felder."""
    nachbar = []
    for p in range(9):
        for q in range(9):
            nachbar.append('true' if q in NACHBARFELDER[p] else 'false')
    start = [int(z) for z in zustand]
    return 'T = %d;\nstart = %s;\nnachbar = array2d(1..9, 1..9, [%s]);' % (zuege, start, ', '.join(nachbar))


print(als_dzn('012368475', 14)[:120], '...')

T = 14;
start = [0, 1, 2, 3, 6, 8, 4, 7, 5];
nachbar = array2d(1..9, 1..9, [false, true, false, true, false, false, fals ...


Beim Erzeugen dieses Notebooks wurde das Modell mit dem Solver **Chuffed** (in der MiniZinc IDE über *Solver configuration* wählbar) für vier unserer Beispiele mit je zwei Werten von `T` ausgeführt. Jede gefundene Lückenfolge wurde in Python **nachgespielt** und führt zum Ziel, und jedes Ergebnis passt zu den wahren Abständen aus Aufgabe 2.2:

| Beispiel | wahrer Abstand | `T` | Ergebnis |
|---|---|---|---|
| A | 10 | 10 | Lösung gefunden |
| A | 10 | 8 | keine Lösung (bewiesen) |
| B | 16 | 16 | Lösung gefunden |
| B | 16 | 14 | keine Lösung (bewiesen) |
| C | 22 | 22 | Lösung gefunden |
| C | 22 | 20 | keine Lösung (bewiesen) |
| E | 31 | 31 | Lösung gefunden |
| E | 31 | 29 | keine Lösung (bewiesen) |

Die Ausgabe für B mit `T = 16` (die Folge der Felder, auf denen die Lücke nacheinander steht):

**Ausgabe:**

```
p = [5, 2, 3, 6, 5, 4, 7, 8, 9, 6, 3, 2, 1, 4, 5, 6, 9]
```

**Was fällt auf?**

- `T = wahrer Abstand` ist lösbar. Mit **zwei Zügen weniger** meldet der Solver `UNSATISFIABLE`: Er **beweist**, dass es nicht kürzer geht. Man sucht also die kleinste Lösung, indem man `T` erhöht (oder vermindert), das ist das iterative Vertiefen aus Abschnitt 5 mit dem Solver als Tiefensuche. Auch das Ergebnis `T = Abstand + 1` wäre unlösbar, wegen der Parität (jeder Zug ändert die Farbe der Lücke auf dem Schachbrett).
- Bei 3 × 3 Feldern ist das Modell überraschend schnell, auch beim schwersten Zustand E. Es ist aber **nur ein kleiner Raum**. Eine Folge von `T` Zügen hat `T` Zeitscheiben mit je 9 Feldern. Wird `T` oder das Brett groß, wächst das Modell und der Solver bleibt stecken. Bei den Pfannkuchen und den Labyrinthen in den Lösungsnotebooks ist das bei den größeren Beispielen der Fall.
- Der Solver beantwortet **eine Frage pro Lauf** („gibt es einen Weg mit genau T Zügen?“). Die Suche in Python mit A\* liefert den **kürzesten Weg** direkt. MiniZinc lohnt sich, wenn zum Zustandsraum **weitere Bedingungen** kommen (zum Beispiel „darf Feld X nie belegt sein“, „zwei Figuren dürfen nie nebeneinander stehen“), die man in Python mühsam einbauen müsste.

## Das hast du gelernt

```python
# A* mit Funktion als Argument: schaetzung(zustand) darf nie größer sein als die wahre Restlänge
def astern_allg(start, nachfolger, ist_ziel, schaetzung):
    beste = {start: 0}
    warteschlange = [(schaetzung(start), 0, start)]       # (f, g, zustand) mit f = g + h
    while warteschlange:
        f, g, zustand = heapq.heappop(warteschlange)
        if g > beste[zustand]:
            continue                                        # veralteter Eintrag
        if ist_ziel(zustand):                               # Zieltest erst beim Herausnehmen
            return g
        for neu in nachfolger(zustand):
            if neu not in beste or g + 1 < beste[neu]:
                beste[neu] = g + 1
                heapq.heappush(warteschlange, (g + 1 + schaetzung(neu), g + 1, neu))
```

- **Zustand und Größe:** Speichere Zustände unveränderlich und **kompakt** (Text, Tupel, Zahl; Vorsicht mit führenden Nullen), in **Normalform**, wenn verschiedene Beschreibungen dasselbe bedeuten (Pfannkuchen normieren). **Schätze die Größe** des Raums, bevor du loslegst (9! = 362 880, 16! ≈ 2 · 10<sup>13</sup>), und prüfe mit einer **Invariante**, ob der Fall überhaupt lösbar ist.
- **Zähle bearbeitete Zustände** und vergleiche Verfahren in einer Tabelle (Verfahren × Beispiel). Das ist unabhängig vom Computer und vom Browser.
- **Vom Ziel aus rechnen:** Eine Breitensuche vom Ziel aus liefert **alle** Abstände auf einmal; dann ist jede Anfrage ein Nachschlagen. Für Aufgaben mit vielen Anfragen oder als **Schätzung** für ein größeres Problem (Abstände der einzelnen Personen bei den Labyrinthen).
- **Suche von beiden Seiten:** etwa 2 · b<sup>k/2</sup> statt b<sup>k</sup> Zustände. Nur, wenn man auch **rückwärts** gehen kann und das Ziel ein einzelner Zustand ist.
- **A\***: Priorität f = g + h mit **zulässiger** Schätzung h (nie größer als die Wahrheit). Der Zieltest erst beim Herausnehmen. Eine **bessere** zulässige Schätzung bearbeitet weniger Zustände. Eine **zu große** Schätzung macht aus A\* eine Heuristik (kein garantiert kürzester Weg); dann muss man die Güte messen.
- **Tiefenschranke und iteratives Vertiefen** brauchen fast keinen Speicher, bearbeiten aber Zustände mehrfach. **IDA\*** verbindet das mit der Schätzung.
- **MiniZinc** kann Planungsaufgaben mit **fester Länge** beantworten und beweist, dass es nicht kürzer geht. Bei längeren Plänen wird das Modell schnell zu groß.

### Aufgaben der 2. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Pancake Sort](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/41_PancakeSort/pancake_loesung.ipynb) (41. BwInf): Normieren, Breitensuche, A\* mit Breakpoints und P(n) bis P(15)
- [Simultane Labyrinthe](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/43_SimultaneLabyrinthe/simultanelabyrinthe_loesung.ipynb) (43.): Paare von Feldern, Abstände vom Ziel aus, die großen Labyrinthe mit numpy
- [Seilschaften](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/33_Seilschaften/seilschaften_loesung.ipynb) (33.): Zustände und Breitensuche
- [Transformers](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/44_Transformers/transformers_loesung.ipynb) (44.) und [Stromrallye](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/38_Stromrallye/stromrallye_loesung.ipynb) (38.): Zustandsräume mit Zusatzbedingungen
- [Rechtsrum in Rechthausen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/35_RechtsrumInRechthausen/rechthausen_loesung.ipynb) (35.): ein größerer Zustandsraum
- [Schach dem Wildschwein](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/37_SchachDemWildschwein/schach_loesung.ipynb) (37.): Gewinnstellungen mit Tiefenschranke und Rückwärtsanalyse (für Schnelle)

Weitere Aufgaben stehen in der Werkzeugübersicht der 2. Runde (Abschnitt „Vollständige Suche bei großen Fällen“). Als Nächstes folgen die Kern-Notebooks **16 „Experimente auswerten“** und **17 „Dokumentation der 2. Runde“**; die Themen-Notebooks (05 und 08 bis 15) kommen danach.